> **运行环境准备**：在执行后续代码之前，先完成两项基础设置。
> 1. `warnings.filterwarnings("ignore")`：关闭运行时的告警信息，避免 LangChain 旧版 API（如 `langchain_classic`）的 deprecation warning 干扰教学阅读。
> 2. 清理 `HTTP_PROXY` / `HTTPS_PROXY` / `ALL_PROXY` 等代理环境变量：防止本地或容器内的代理配置影响 Ollama、Milvus、SiliconFlow 等服务的正常访问。

In [1]:
import warnings
warnings.filterwarnings("ignore")


import os
for k in [
    "HTTP_PROXY",
    "HTTPS_PROXY",
    "ALL_PROXY",
    "http_proxy",
    "https_proxy",
    "all_proxy"
]:
    os.environ.pop(k, None)

# 检索增强生成 RAG 简介

## 什么是 RAG

大型语言模型（LLM）相较于传统的语言模型具有更强大的能力，然而在某些情况下，它们仍可能无法提供准确的答案。为了解决大型语言模型在生成文本时面临的一系列挑战，提高模型的性能和输出质量，研究人员提出了一种新的模型架构：**检索增强生成（RAG, Retrieval-Augmented Generation）**。该架构巧妙地**整合了从庞大知识库中检索到的相关信息，并以此为基础，指导大型语言模型生成更为精准的答案**，从而显著提升了回答的准确性与深度。

目前 LLM 面临的主要问题有：

- **信息偏差/幻觉：** LLM 有时会产生与客观事实不符的信息，导致用户接收到的信息不准确。RAG 通过检索数据源，辅助模型生成过程，确保输出内容的精确性和可信度，减少信息偏差。

- **知识更新滞后性：** LLM 基于静态的数据集训练，这可能导致模型的知识更新滞后，无法及时反映最新的信息动态。RAG 通过实时检索最新数据，保持内容的时效性，确保信息的持续更新和准确性。

- **内容不可追溯：** LLM 生成的内容往往缺乏明确的信息来源，影响内容的可信度。RAG 将生成内容与检索到的原始资料建立链接，增强了内容的可追溯性，从而提升了用户对生成内容的信任度。

- **领域专业知识能力欠缺：** LLM 在处理特定领域的专业知识时，效果可能不太理想，这可能会影响到其在相关领域的回答质量。RAG 通过检索特定领域的相关文档，为模型提供丰富的上下文信息，从而提升了在专业领域内的问题回答质量和深度。

- **推理能力限制：** 面对复杂问题时，LLM 可能缺乏必要的推理能力，这影响了其对问题的理解和回答。RAG 结合检索到的信息和模型的生成能力，通过提供额外的背景知识和数据支持，增强了模型的推理和理解能力。

- **应用场景适应性受限：** LLM 需在多样化的应用场景中保持高效和准确，但单一模型可能难以全面适应所有场景。RAG 使得 LLM 能够通过检索对应应用场景数据的方式，灵活适应问答系统、推荐系统等多种应用场景。

- **长文本处理能力较弱：** LLM 在理解和生成长篇内容时受限于有限的上下文窗口，且必须按顺序处理内容，输入越长，速度越慢。RAG 通过检索和整合长文本信息，强化了模型对长上下文的理解和生成，有效突破了输入长度的限制，同时降低了调用成本，并提升了整体的处理效率。

## RAG 的工作流程

RAG 是一个完整的系统，其工作流程可以简单地分为数据处理、检索、增强和生成四个阶段：

<img src="assets/rag-02-10.png" width="100%">

1. **数据处理阶段**
   1. 对原始数据进行清洗和处理。
   2. 将处理后的数据转化为检索模型可以使用的格式。
   3. 将处理后的数据存储在对应的数据库中。
2. **检索阶段**
   1. 将用户的问题输入到检索系统中，从数据库中检索相关信息。
3. **增强阶段**
   1. 对检索到的信息进行处理和增强，以便生成模型可以更好地理解和使用。
4. **生成阶段**
   1. 将增强后的信息输入到生成模型中，生成模型根据这些信息生成答案。

## RAG VS Finetune

在提升大语言模型效果中，RAG 和 微调（Finetune）是两种主流的方法。

**微调**: 通过在特定数据集上进一步训练大语言模型，来提升模型在特定任务上的表现。

RAG 和 微调的对比可以参考下表（表格来源[[1](https://arxiv.org/abs/2312.10997)][[2](https://baoyu.io/translations/ai-paper/2312.10997-retrieval-augmented-generation-for-large-language-models-a-survey)]）

| 特征比较 | RAG                                                          | 微调                                                         |
| -------- | ------------------------------------------------------------ | ------------------------------------------------------------ |
| 知识更新 | 直接更新检索知识库，无需重新训练。信息更新成本低，适合动态变化的数据。 | 通常需要重新训练来保持知识和数据的更新。更新成本高，适合静态数据。 |
| 外部知识 | 擅长利用外部资源，特别适合处理文档或其他结构化/非结构化数据库。 | 将外部知识学习到 LLM 内部。                                  |
| 数据处理 | 对数据的处理和操作要求极低。                                 | 依赖于构建高质量的数据集，有限的数据集可能无法显著提高性能。 |
| 模型定制 | 侧重于信息检索和融合外部知识，但可能无法充分定制模型行为或写作风格。 | 可以根据特定风格或术语调整 LLM 行为、写作风格或特定领域知识。 |
| 可解释性 | 可以追溯到具体的数据来源，有较好的可解释性和可追踪性。       | 黑盒子，可解释性相对较低。                                   |
| 计算资源 | 需要额外的资源来支持检索机制和数据库的维护。                 | 依赖高质量的训练数据集和微调目标，对计算资源的要求较高。     |
| 推理延迟 | 增加了检索步骤的耗时                                         | 单纯 LLM 生成的耗时                                          |
| 降低幻觉 | 通过检索到的真实信息生成回答，降低了产生幻觉的概率。         | 模型学习特定领域的数据有助于减少幻觉，但面对未见过的输入时仍可能出现幻觉。 |
| 伦理隐私 | 检索和使用外部数据可能引发伦理和隐私方面的问题。             | 训练数据中的敏感信息需要妥善处理，以防泄露。                 |

## LangChain


### 什么是 LangChain

ChatGPT 的巨大成功激发了越来越多的开发者兴趣，他们希望利用 OpenAI 提供的 API 或者私有化模型，来开发基于大型语言模型的应用程序。尽管大型语言模型的调用相对简单，但要创建完整的应用程序，仍然需要大量的定制开发工作，包括 API 集成、互动逻辑、数据存储等等。

为了解决这个问题，从 2022 年开始，许多机构和个人相继推出了多个开源项目，旨在**帮助开发者们快速构建基于大型语言模型的端到端应用程序或工作流程**。其中一个备受关注的项目就是 LangChain 框架。

**LangChain 框架是一个开源工具，充分利用了大型语言模型的强大能力，以便开发各种下游应用。它的目标是为各种大型语言模型应用提供通用接口，从而简化应用程序的开发流程**。具体来说，LangChain 框架可以实现数据感知和环境互动，也就是说，它能够让语言模型与其他数据来源连接，并且允许语言模型与其所处的环境进行互动。

利用 LangChain 框架，我们可以轻松地构建如下所示的 RAG 应用。在下图中，`每个椭圆形代表了 LangChain 的一个模块`，例如数据收集模块或预处理模块。`每个矩形代表了一个数据状态`，例如原始数据或预处理后的数据。箭头表示数据流的方向，从一个模块流向另一个模块。在每一步中，LangChain 都可以提供对应的解决方案，帮助我们处理各种任务。

![Langchain 示意图](assets/C1-3-langchain.png)

### LangChain 的核心组件

LangChian 作为一个大语言模型开发框架，可以将 LLM 模型（对话模型、embedding 模型等）、向量数据库、交互层 Prompt、外部知识、外部代理工具整合到一起，进而可以自由构建 LLM 应用。 LangChain 主要由以下 6 个核心组件组成:

- **模型输入/输出（Model I/O）**：与语言模型交互的接口
- **数据连接（Data connection）**：与特定应用程序的数据进行交互的接口
- **链（Chains）**：将组件组合实现端到端应用。比如后续我们会将搭建`检索问答链`来完成检索问答。
- **记忆（Memory）**：用于链的多次运行之间持久化应用程序状态；
- **代理（Agents）**：扩展模型的推理能力。用于复杂的应用的调用序列；
- **回调（Callbacks）**：扩展模型的推理能力。用于复杂的应用的调用序列；

# 项目说明

本项目是一个面向初学者的**检索增强生成（RAG）入门实践**，以 LangChain 为核心框架，完整演示如何从零搭建一个"本地知识库问答"系统。项目将带你依次了解 RAG 的基本原理、环境准备、文档加载与切分、向量化与向量存储、检索策略、大模型生成，以及简单的 LLM-as-Judge 评估方法。

## 项目总览

整个 Notebook 按照"数据 → 索引 → 检索 → 生成 → 评估"的链路组织，帮助学习者建立 RAG 系统的端到端视角。主要内容包括：

1. **环境准备**：介绍 Conda / UV 虚拟环境搭建、依赖库安装及 Ollama 本地模型配置；
2. **知识库构建**：使用 LangChain 加载 PDF、Markdown 等本地文档，进行文本切分、嵌入向量化，并存储到向量数据库；
3. **检索与生成**：基于向量检索召回相关片段，结合本地大语言模型生成带有上下文的回答；
4. **评估闭环**：通过 LLM-as-Judge 对比 RAG 回答与基线回答的忠实度、完整性和相关性；
5. **扩展方向**：简要介绍 Ragas 等系统化评估框架，为后续优化提供参考。

学习完本项目后，你将能够独立搭建一个可运行、可评估的本地知识库问答系统，并理解 RAG 链路中每一步的作用与调优点。

## 本项目涉及的关键库与环境说明

> **环境说明**：本 Notebook 在本地 `env_rag` Conda 环境中运行，Python 版本为 3.12，Jupyter Kernel 名称为 `Python3 (env_rag)`。下面列出本项目核心依赖库、用途说明以及 `requirements.txt` 中对应的版本号。实际运行时，也可以通过下方代码单元查看当前环境中各库的真实安装版本。

| 库名 | 版本（requirements.txt） | 说明 |
|---|---|---|
| `langchain` | 1.3.9 | LangChain 主框架，提供链、提示词、模型接口等核心能力 |
| `langchain-core` | 1.4.7 | LangChain 核心抽象与运行时（Messages、Runnables、输出解析等） |
| `langchain-community` | 0.4.2 | 社区版集成（如 `PyMuPDFLoader` 等文档加载器） |
| `langchain-ollama` | 1.1.0 | Ollama 本地模型集成（Embedding、Chat、Reranker） |
| `langchain-milvus` | 0.3.3 | Milvus 向量数据库的 LangChain 封装 |
| `langchain-classic` | 1.0.8 | 兼容旧版 API（`RetrievalQA`、`ConversationalRetrievalChain`） |
| `langchain-text-splitters` | 1.1.2 | 文本切分器（`RecursiveCharacterTextSplitter` 等） |
| `langchain-cohere` | 0.6.0 | Cohere Rerank 云端精排（可选） |
| `langchain-openai` | 1.3.2 | OpenAI 兼容接口（本项目中用于 SiliconFlow） |
| `pymilvus` | 2.6.15 | Milvus Python SDK，用于连接与操作向量库 |
| `PyMuPDF` | 1.26.4 | PDF 解析器，`PyMuPDFLoader` 的底层依赖 |
| `rank-bm25` | 0.2.2 | 教学用手写 BM25 关键词检索 |
| `sentence-transformers` | 5.5.1 | 本地 Embedding/Reranker 模型支持 |
| `python-dotenv` | 1.2.2 | 从 `.env` 文件读取 API Key 等环境变量 |
| `httpx` | 0.28.1 | HTTP 客户端，用于 SiliconFlow / Jina Rerank 等 API 调用 |
| `openai` | 2.41.1 | OpenAI SDK，SiliconFlow 兼容调用所需 |
| `ollama` | 0.6.2 | Ollama Python 客户端 |
| `cohere` | 5.21.1 | Cohere 官方 SDK（可选 Rerank） |
| `pandas` | 2.3.3 | 数据处理与展示 |
| `numpy` | 1.26.4 | 数值计算 |
| `ipykernel` | 7.3.0 | Jupyter Notebook 运行所需 Kernel |

In [2]:
# 查看当前 env_rag 环境中关键库的实际安装版本
from importlib.metadata import version, PackageNotFoundError

packages = [
    ("langchain", "LangChain 主框架"),
    ("langchain-core", "LangChain 核心抽象"),
    ("langchain-community", "社区版组件与加载器"),
    ("langchain-ollama", "Ollama 本地模型集成"),
    ("langchain-milvus", "Milvus 向量库封装"),
    ("langchain-classic", "旧版 Chain API 兼容"),
    ("langchain-text-splitters", "文本切分器"),
    ("langchain-cohere", "Cohere Rerank（可选）"),
    ("langchain-openai", "OpenAI 兼容接口"),
    ("pymilvus", "Milvus Python SDK"),
    ("PyMuPDF", "PDF 解析"),
    ("rank-bm25", "BM25 关键词检索（教学版）"),
    ("sentence-transformers", "本地 Embedding/Reranker"),
    ("python-dotenv", ".env 环境变量读取"),
    ("httpx", "HTTP 客户端"),
    ("openai", "OpenAI SDK"),
    ("ollama", "Ollama Python 客户端"),
    ("cohere", "Cohere SDK（可选）"),
    ("pandas", "数据处理"),
    ("numpy", "数值计算"),
    ("ipykernel", "Jupyter Kernel"),
]

print(f"{'库名':<35} {'版本':<15} {'说明'}")
print("-" * 80)
for name, desc in packages:
    try:
        ver = version(name)
    except PackageNotFoundError:
        ver = "未安装"
    print(f"{name:<35} {ver:<15} {desc}")


库名                                  版本              说明
--------------------------------------------------------------------------------
langchain                           1.3.9           LangChain 主框架
langchain-core                      1.4.7           LangChain 核心抽象
langchain-community                 0.4.2           社区版组件与加载器
langchain-ollama                    1.1.0           Ollama 本地模型集成
langchain-milvus                    0.3.3           Milvus 向量库封装
langchain-classic                   1.0.8           旧版 Chain API 兼容
langchain-text-splitters            1.1.2           文本切分器
langchain-cohere                    0.6.0           Cohere Rerank（可选）
langchain-openai                    1.3.2           OpenAI 兼容接口
pymilvus                            2.6.15          Milvus Python SDK
PyMuPDF                             1.26.4          PDF 解析
rank-bm25                           0.2.2           BM25 关键词检索（教学版）
sentence-transformers               5.5.1           本地 Embedding/Reranker
python-

## 统一模型配置说明

本 notebook 统一使用以下三个模型配置：

- **Embedding 模型**：通过 `langchain_ollama.OllamaEmbeddings` 调用 `qwen3-embedding:0.6b`，服务地址为 `http://192.168.1.166:11434`。
- **大语言模型**：通过 `langchain.chat_models.init_chat_model` 调用 SiliconFlow 的 `Qwen/Qwen3-8B`，API Key 从仓库根目录 `.env` 中的 `SILICONFLOW_API_KEY` 读取。
- **Reranker 模型**：通过 `langchain_ollama.ChatOllama` 调用 `dengcao/Qwen3-Reranker-0.6B:Q8_0`，服务地址为 `http://192.168.1.166:11434`。

下方统一定义 `my_emb`、`my_llm` 和 reranker 配置常量。后续章节为了便于单独运行，可能会在对应代码单元中重复定义同样配置。


In [3]:
import os
import httpx
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_ollama import OllamaEmbeddings

# ── 1. 大语言模型配置 ─────────────────────────────
env_path = "/Users/will/workspace/llm-rag/.env"
load_dotenv(env_path)
api_key = os.getenv("SILICONFLOW_API_KEY")

my_llm = init_chat_model(
    model="Qwen/Qwen3-8B",
    model_provider="openai",
    base_url="https://api.siliconflow.cn/v1",
    api_key=api_key,
    http_client=httpx.Client(trust_env=False),
    temperature=0.3,
    max_tokens=3000,
)

# ── 2.  Embedding 模型配置 ───────────────────
my_emb = OllamaEmbeddings(
    base_url="http://192.168.1.166:11434",
    model="qwen3-embedding:0.6b",
)

# ── 3. Reranker 模型配置 ───────────────────────
RERANKER_BASE_URL = "http://192.168.1.166:11434"
RERANKER_MODEL = "dengcao/Qwen3-Reranker-0.6B:Q8_0"

# 数据预处理

## 数据简单预处理

为构建我们的本地知识库，我们需要对以多种类型存储的本地文档进行处理，读取本地文档并通过前文描述的 Embedding 方法将本地文档的内容转化为词向量来构建向量数据库。接下来以一些实际示例入手，来讲解如何对本地文档进行处理。
### 数据读取
如果没有知识库文件，可以选择将知识库源数据放置在 knowledge_db 目录下

In [4]:
# !cp -r /workspace/DataSet/rag2026/VMAX-S/   /workspace/ai-showcase/3.3大模型-RAG/knowledge_path/

**PDF文档为例**  
可以使用 LangChain 的 PyMuPDFLoader 来读取知识库的 PDF 文件。PyMuPDFLoader 是 PDF 解析器中速度最快的一种，结果会包含 PDF 及其页面的详细元数据，并且每页返回一个文档。

In [5]:
from langchain_community.document_loaders import PyMuPDFLoader

# 创建一个 PyMuPDFLoader Class 实例，输入为待加载的 pdf 文档路径
loader = PyMuPDFLoader("knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf")

# 调用 PyMuPDFLoader Class 的函数 load 对 pdf 文件进行加载
pdf_pages = loader.load()

文档加载后储存在 `pdf_pages` 变量中:
- `pdf_pages` 的变量类型为 `List`
- 打印 `pdf_pages` 的长度可以看到 pdf 一共包含多少页

In [6]:
print(f"载入后的变量类型为：{type(pdf_pages)}，",  f"该 PDF 一共包含 {len(pdf_pages)} 页")

载入后的变量类型为：<class 'list'>， 该 PDF 一共包含 29 页


`pdf_pages` 中的每一元素为一个文档，变量类型为 `langchain_core.documents.base.Document`, 文档变量类型包含两个属性
- `page_content` 包含该文档的内容。
- `meta_data` 为文档相关的描述性数据。

In [7]:
pdf_page = pdf_pages[20]
print(f"每一个元素的类型：{type(pdf_page)}.", 
    f"该文档的描述性数据：{pdf_page.metadata}", 
    f"查看该文档的内容:\n{pdf_page.page_content}", 
    sep="\n------\n")

每一个元素的类型：<class 'langchain_core.documents.base.Document'>.
------
该文档的描述性数据：{'producer': 'Apache FOP Version 2.6', 'creator': 'DITA Open Toolkit', 'creationdate': '2023-05-23T21:45:33+08:00', 'source': 'knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', 'file_path': 'knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', 'total_pages': 29, 'format': 'PDF 1.4', 'title': '目录', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20230523214533+08'00'", 'page': 20}
------
查看该文档的内容:
7 可靠性设计
容错能力
ZXVMAX-S通过对关键软件资源的定时检测、实时任务监控、存储保护、数据校验、操作日志
信息保存等手段，可有效地防止小软件故障对系统所造成的冲击，提高了软件系统的容错能力
（即软件错误情况下的自愈能力）。
故障监视及处理
ZXVMAX-S具备自动检测与诊断系统软硬件故障的功能，可对故障硬件实施自动隔离、倒换、
重新启动、重新加载等处理。
满足漏洞扫描
为了避免存在安全漏洞使在网设备受外部攻击，需要使系统满足主流安全漏洞扫描工具的要求。
ZXVMAX-S满足Nessus和CD工具安全漏洞扫描的要求。
7.3 数据可靠性
ZXVMAX-S产品在不同分层架构下的数据都可以确保数据可靠和可恢复。

采集层
采集层运行环境部署采用冗余磁盘阵列设计，在服务器损坏、单一硬盘故障等场景下，通过
替换故障硬件后即可工作，数据不会丢失。同时，系统支持数据补充采集，确保系统数据连
续性。

存储共享层
存储共享层的MPP数据库，根据需要都设置了多个冗余备份，单一节点的损坏不会影响整系


In [8]:
# 假设环境中缺少 NLTK 数据包中的 punkt 资源，这是 NLTK 库中用于分词的一个重要组件。解决这个问题的方法是按照报错信息中提到的步骤下载并安装 punkt 资源。
# import nltk
# nltk.download('punkt', quiet=True)

### 数据清洗

简单版本只做通用空白归一化：把连续换行和空格压缩为一个空格。它容易理解，但不会识别页眉页脚、标题、列表和 PDF 排版硬换行，作为后续优化版本的基线。

In [9]:
import re
from langchain_core.documents import Document

def simple_clean_document(doc: Document) -> Document:
    text = re.sub(r"\s+", " ", doc.page_content).strip()
    metadata = dict(doc.metadata)
    metadata["page_number"] = metadata.get("page", 0) + 1
    metadata["preprocess_version"] = "simple"
    return Document(page_content=text, metadata=metadata)

simple_cleaned_pages = [
    simple_clean_document(page) for page in pdf_pages if page.page_content.strip()
]
print(simple_cleaned_pages[20].page_content[:500])

7 可靠性设计 容错能力 ZXVMAX-S通过对关键软件资源的定时检测、实时任务监控、存储保护、数据校验、操作日志 信息保存等手段，可有效地防止小软件故障对系统所造成的冲击，提高了软件系统的容错能力 （即软件错误情况下的自愈能力）。 故障监视及处理 ZXVMAX-S具备自动检测与诊断系统软硬件故障的功能，可对故障硬件实施自动隔离、倒换、 重新启动、重新加载等处理。 满足漏洞扫描 为了避免存在安全漏洞使在网设备受外部攻击，需要使系统满足主流安全漏洞扫描工具的要求。 ZXVMAX-S满足Nessus和CD工具安全漏洞扫描的要求。 7.3 数据可靠性 ZXVMAX-S产品在不同分层架构下的数据都可以确保数据可靠和可恢复。  采集层 采集层运行环境部署采用冗余磁盘阵列设计，在服务器损坏、单一硬盘故障等场景下，通过 替换故障硬件后即可工作，数据不会丢失。同时，系统支持数据补充采集，确保系统数据连 续性。  存储共享层 存储共享层的MPP数据库，根据需要都设置了多个冗余备份，单一节点的损坏不会影响整系 统的运行，也不会导致数据丢失。在替换损坏硬件后，系统能自动进行数据冗余备份的补全 和负载均衡


### 文档分割

简单版本直接按固定字符长度切分，不感知章节边界。

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

simple_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50,
)
simple_split_docs = simple_text_splitter.split_documents(simple_cleaned_pages)
for index, chunk in enumerate(simple_split_docs):
    chunk.metadata["chunk_index"] = index

print(f"简单预处理切分后的文件数量：{len(simple_split_docs)}")

简单预处理切分后的文件数量：54


## 数据预处理优化

前面的内容演示了 PDF 的基本加载与页面结构。本节在此基础上给出面向 ZXVMAX-S 产品说明书的优化处理流程：先恢复文档结构并清除页眉页脚等噪声，再按章节标题建立语义边界，最后仅对过长章节进行二次切分，同时保留来源、页码、业务类型、章节标题和 chunk 序号等元数据。

### 数据读取

优化版本读取同一份 PDF，确保后续对比只反映预处理策略的差异。

In [11]:
optimized_loader = PyMuPDFLoader(
    "knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf"
)
optimized_pdf_pages = optimized_loader.load()
print(f"优化版本读取页数：{len(optimized_pdf_pages)}")

优化版本读取页数：29


### 数据清洗

PDF 清洗不是简单地删除换行和空格，而是尽可能恢复文档原有的语义结构。对于本项目的 ZXVMAX-S 产品说明书，需要重点处理：页眉页脚、独立页码、PDF 排版产生的硬换行，以及 `` 等私有区项目符号。  

本节按三个步骤进行：先规范字符，再按规则过滤噪声，最后恢复段落结构。标题、列表、英文术语和必要空格会被保留。

定义可复用的清洗规则

In [12]:
import re
import unicodedata
from pathlib import Path
from langchain_core.documents import Document

# ZXVMAX-S 文档的固定页脚，例如：SJ-... | 2023-03-30（R1.0）
FOOTER_PATTERN = re.compile(
    r"^SJ-\d{14,}-\d+\s*\|\s*\d{4}-\d{2}-\d{2}（R\d+\.\d+）$"
)
PAGE_NUMBER_PATTERN = re.compile(r"^\d{1,4}$")
HEADING_PATTERN = re.compile(
    r"^(?:第[一二三四五六七八九十百]+章|\d+(?:\.\d+)*)(?:\s+|\u00a0).+"
)
BULLET_PATTERN = re.compile(r"^[\uf06c•▪▫▶●○]\s*")
REPEATED_HEADER_PATTERN = re.compile(
    r"^ZXVMAX-S(?: 多维价值分析系统)? 产品描述（.+业务）$"
)

In [13]:
def normalize_line(line: str) -> str:
    """统一 Unicode、空白和项目符号，不删除有语义的空格。"""
    # NFC 只统一 Unicode 组合形式，不会把中文标点转换成英文标点
    line = unicodedata.normalize("NFC", line)
    line = line.replace("\u00a0", " ").replace("\u3000", " ")
    line = re.sub(r"[ \t]+", " ", line).strip()
    if BULLET_PATTERN.match(line):
        line = BULLET_PATTERN.sub("- " , line)
    return line


def should_join(previous: str, current: str) -> bool:
    """判断换行是否由 PDF 排版造成；标题、列表和完整句子不合并。"""
    if not previous or not current:
        return False
    if current.startswith("- ") or previous.startswith("- "):
        return False
    if HEADING_PATTERN.match(previous) or HEADING_PATTERN.match(current):
        return False
    # 产品说明书中的无编号小节标题通常较短，如“容错能力”“采集层”
    if len(previous) <= 20 and not previous.endswith(("，", "、", "：", "；")):
        return False
    if previous.endswith(("。", "！", "？", "；", "：", "）", "】")):
        return False
    return True


def clean_pdf_document(doc: Document) -> Document:
    """清洗一个 PDF 页面，同时完整保留来源和页码 metadata。"""
    raw_text = doc.page_content.replace("\r\n", "\n").replace("\r", "\n")
    lines = []
    pending_bullet = False

    for raw_line in raw_text.split("\n"):
        line = normalize_line(raw_line)
        if not line:
            continue
        # PDF 中项目符号经常单独占一行，把它与下一行的列表标题合并
        if line == "- ":
            pending_bullet = True
            continue
        if pending_bullet:
            line = "- " + line
            pending_bullet = False
        if FOOTER_PATTERN.match(line) or PAGE_NUMBER_PATTERN.match(line):
            continue
        if REPEATED_HEADER_PATTERN.match(line):
            continue
        lines.append(line)

    paragraphs = []
    for line in lines:
        if paragraphs and should_join(paragraphs[-1], line):
            paragraphs[-1] += line
        else:
            paragraphs.append(line)

    metadata = dict(doc.metadata)
    metadata["source_name"] = Path(metadata.get("source", "")).name
    metadata["page_number"] = metadata.get("page", 0) + 1
    metadata["cleaned"] = True

    return Document(
        page_content="\n".join(paragraphs).strip(),
        metadata=metadata,
    )


# 对优化版本的全部页面清洗；保留原始页面方便做前后对照
cleaned_pdf_pages = [
    clean_pdf_document(page)
    for page in optimized_pdf_pages
    if page.page_content.strip()
]
cleaned_pdf_page = cleaned_pdf_pages[20]

# 清洗前后对照与基础质量检查
print("===== 清洗前（前 500 字）=====")
print(optimized_pdf_pages[20].page_content[:500])
print("\n===== 清洗后（前 500 字）=====")
print(cleaned_pdf_page.page_content[:500])
print("\n===== 质量统计 =====")
print("原始页数：", len(optimized_pdf_pages))
print("清洗后有效页数：", len(cleaned_pdf_pages))
print("原始字符数：", sum(len(page.page_content) for page in optimized_pdf_pages))
print("清洗后字符数：", sum(len(page.page_content) for page in cleaned_pdf_pages))
print("残留私有项目符号页数：", sum("\uf06c" in page.page_content for page in cleaned_pdf_pages))
print("当前页元数据：", cleaned_pdf_page.metadata)

===== 清洗前（前 500 字）=====
7 可靠性设计
容错能力
ZXVMAX-S通过对关键软件资源的定时检测、实时任务监控、存储保护、数据校验、操作日志
信息保存等手段，可有效地防止小软件故障对系统所造成的冲击，提高了软件系统的容错能力
（即软件错误情况下的自愈能力）。
故障监视及处理
ZXVMAX-S具备自动检测与诊断系统软硬件故障的功能，可对故障硬件实施自动隔离、倒换、
重新启动、重新加载等处理。
满足漏洞扫描
为了避免存在安全漏洞使在网设备受外部攻击，需要使系统满足主流安全漏洞扫描工具的要求。
ZXVMAX-S满足Nessus和CD工具安全漏洞扫描的要求。
7.3 数据可靠性
ZXVMAX-S产品在不同分层架构下的数据都可以确保数据可靠和可恢复。

采集层
采集层运行环境部署采用冗余磁盘阵列设计，在服务器损坏、单一硬盘故障等场景下，通过
替换故障硬件后即可工作，数据不会丢失。同时，系统支持数据补充采集，确保系统数据连
续性。

存储共享层
存储共享层的MPP数据库，根据需要都设置了多个冗余备份，单一节点的损坏不会影响整系
统的运行，也不会导致数据丢失。在替换损坏硬件后，系统能自动进行数据冗余备份的补全
和负载均衡

===== 清洗后（前 500 字）=====
7 可靠性设计
容错能力
ZXVMAX-S通过对关键软件资源的定时检测、实时任务监控、存储保护、数据校验、操作日志信息保存等手段，可有效地防止小软件故障对系统所造成的冲击，提高了软件系统的容错能力（即软件错误情况下的自愈能力）。
故障监视及处理
ZXVMAX-S具备自动检测与诊断系统软硬件故障的功能，可对故障硬件实施自动隔离、倒换、重新启动、重新加载等处理。
满足漏洞扫描
为了避免存在安全漏洞使在网设备受外部攻击，需要使系统满足主流安全漏洞扫描工具的要求。
ZXVMAX-S满足Nessus和CD工具安全漏洞扫描的要求。
7.3 数据可靠性
ZXVMAX-S产品在不同分层架构下的数据都可以确保数据可靠和可恢复。
- 采集层
采集层运行环境部署采用冗余磁盘阵列设计，在服务器损坏、单一硬盘故障等场景下，通过替换故障硬件后即可工作，数据不会丢失。同时，系统支持数据补充采集，确保系统数据连续性。
- 存储共享层
存储共享层的MPP数据库，根据需要都设置了多个冗余备份，单一节点的损坏不会影响整系统的

### 文档分割

由于单个文档的长度往往会超过模型支持的上下文，导致检索得到的知识太长超出模型的处理能力，因此，在构建向量知识库的过程中，我们往往需要对文档进行分割，将单个文档按长度或者按固定的规则分割成若干个 chunk，然后将每个 chunk 转化为词向量，存储到向量数据库中。

在检索时，我们会以 chunk 作为检索的元单位，也就是每一次检索到 k 个 chunk 作为模型可以参考来回答用户问题的知识，这个 k 是我们可以自由设定的。

Langchain 中文本分割器都根据 `chunk_size` (块大小)和 `chunk_overlap` (块与块之间的重叠大小)进行分割。

![image.png](assets/rag8.png)

* chunk_size 指每个块包含的字符或 Token （如单词、句子等）的数量

* chunk_overlap 指两个块之间共享的字符数量，用于保持上下文的连贯性，避免分割丢失上下文信息

**Langchain 提供多种文档分割方式，区别在怎么确定块与块之间的边界、块由哪些字符/token组成、以及如何测量块大小**

- RecursiveCharacterTextSplitter(): 按字符串分割文本，递归地尝试按不同的分隔符进行分割文本。
- CharacterTextSplitter(): 按字符来分割文本。
- MarkdownHeaderTextSplitter(): 基于指定的标题来分割markdown 文件。
- TokenTextSplitter(): 按token来分割文本。
- SentenceTransformersTokenTextSplitter(): 按token来分割文本
- Language(): 用于 CPP、Python、Ruby、Markdown 等。
- NLTKTextSplitter(): 使用 NLTK（自然语言工具包）按句子分割文本。
- SpacyTextSplitter(): 使用 Spacy按句子的切割文本。

In [14]:
''' 
* RecursiveCharacterTextSplitter 递归字符文本分割
RecursiveCharacterTextSplitter 将按不同的字符递归地分割(按照这个优先级["\n\n", "\n", " ", ""])，
    这样就能尽量把所有和语义相关的内容尽可能长时间地保留在同一位置
RecursiveCharacterTextSplitter需要关注的是4个参数：

* separators - 分隔符字符串数组
* chunk_size - 每个文档的字符数量限制
* chunk_overlap - 两份文档重叠区域的长度
* length_function - 长度计算函数
'''
#导入文本分割器
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [15]:
# 知识库中单段文本长度
CHUNK_SIZE = 512

# 知识库中相邻文本重合长度
OVERLAP_SIZE = 50

In [16]:
# 第二阶段只负责切分过长章节；第一阶段的章节边界由下面的函数识别
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=OVERLAP_SIZE,
    separators=["\n\n", "\n", "。", "；", "！", "？", "，", ""],
)

SECTION_TITLE_PATTERN = re.compile(
    r"^(?:第[一二三四五六七八九十百]+章(?:\s+.+)?|\d+(?:\.\d+)+(?:\s+.+)|\d+\s+[^0-9].+)"
)
BUSINESS_TYPE_PATTERN = re.compile(r"（([^（）]+业务)）")


def is_section_title(line: str) -> bool:
    # 目录条目通常包含连续点线，不应当作为正文的章节边界
    return not re.search(r"[.．·…]{3,}", line) and bool(SECTION_TITLE_PATTERN.match(line))


def extract_business_type(source_name: str) -> str:
    match = BUSINESS_TYPE_PATTERN.search(source_name)
    return match.group(1) if match else "未知业务"


def build_section_documents(pages):
    """先按文档和页面识别章节；跨页时继承上一页最后识别到的章节。"""
    section_docs = []
    current_section_by_source = {}

    for page in pages:
        metadata = dict(page.metadata)
        source_name = metadata.get("source_name") or Path(metadata.get("source", "")).name
        current_section = current_section_by_source.get(source_name, "未识别章节")
        buffer = []

        def flush_buffer():
            if not buffer:
                return
            block_text = "\n".join(buffer).strip()
            # 页尾可能只有下一章节标题，正文会出现在下一页；避免产生标题空块
            if block_text == current_section:
                buffer.clear()
                return
            block_metadata = dict(metadata)
            block_metadata.update({
                "source_name": source_name,
                "business_type": extract_business_type(source_name),
                "section_title": current_section,
            })
            section_docs.append(Document(
                page_content=block_text,
                metadata=block_metadata,
            ))
            buffer.clear()

        for line in page.page_content.splitlines():
            if is_section_title(line):
                flush_buffer()
                current_section = line
                current_section_by_source[source_name] = current_section
            buffer.append(line)
        flush_buffer()

    return section_docs


def split_documents_by_section(pages):
    """章节优先、长度兜底切分，并为每个来源生成稳定的 chunk_index。"""
    from collections import defaultdict

    section_docs = build_section_documents(pages)
    chunks = text_splitter.split_documents(section_docs)
    source_counters = defaultdict(int)

    for chunk in chunks:
        source_name = chunk.metadata["source_name"]
        chunk.metadata["chunk_index"] = source_counters[source_name]
        source_counters[source_name] += 1

        # 标题也进入 embedding/BM25 文本，提升章节名称相关问题的召回率
        section_title = chunk.metadata["section_title"]
        if section_title != "未识别章节" and not chunk.page_content.startswith(section_title):
            chunk.page_content = f"{section_title}\n{chunk.page_content}"

    return chunks

In [17]:
# 先按章节标题分段，再对超过 CHUNK_SIZE 的章节做二次切分
split_docs = split_documents_by_section(cleaned_pdf_pages)
print(f"章节感知切分后的文件数量：{len(split_docs)}")
print("metadata 示例：", {
    key: split_docs[0].metadata.get(key)
    for key in ["source_name", "page_number", "business_type", "section_title", "chunk_index"]
})

章节感知切分后的文件数量：83
metadata 示例： {'source_name': 'ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', 'page_number': 1, 'business_type': '上网日志业务', 'section_title': '未识别章节', 'chunk_index': 0}


In [18]:
split_docs[50].page_content

'5.1 典型组网应用\nZXVMAX-S典型组网如图5-1所示。\n图5-1 ZXVMAX IPDR典型组网应用1. 探针Prober要求部署在各个被采集单元所在的机房，便于采集设备的部署，减少流量负担。\n2. 防火墙日志处理单元要求在防火墙所在的机房，便于防火墙日志，减少流量负担。\n3. 数据合成单元、数据存储单元、应用服务器单元部署在同一个机房，一般部署在中心机房。'

In [19]:
required_metadata = {"source_name", "page_number", "business_type", "section_title", "chunk_index"}
print(f"切分后的字符数：{sum(len(doc.page_content) for doc in split_docs)}")
print("metadata 是否完整：", all(
    required_metadata.issubset(doc.metadata) for doc in split_docs
))
print("误识别为标题的目录点线数：", sum(
    bool(re.search(r"[.．·…]{3,}", doc.metadata["section_title"]))
    for doc in split_docs
))

切分后的字符数：17172
metadata 是否完整： True
误识别为标题的目录点线数： 0


注：文档分割往往决定检索系统的下限。本节采用两阶段策略：先按编号章节标题恢复语义边界，再用 `chunk_size` 和 `chunk_overlap` 对过长章节做二次切分。目录中的连续点线会被排除，跨页正文会继承上一页的章节标题。每个 chunk 同时保存来源、页码、业务类型、章节标题和文档内序号，便于过滤检索、引用溯源和问题定位。

### 简单预处理与优化预处理对比

下面固定比较同一份文档第 21 页中包含“7.3 数据可靠性”的数据块。简单版本按长度截取，优化版本按章节组织，并携带更完整的检索与溯源元数据。

In [20]:
COMPARE_PAGE_NUMBER = 21
COMPARE_SECTION = "7.3 数据可靠性"

simple_example = next(
    doc for doc in simple_split_docs
    if doc.metadata.get("page_number") == COMPARE_PAGE_NUMBER
    and "数据可靠性" in doc.page_content
)
optimized_example = next(
    doc for doc in split_docs
    if doc.metadata.get("page_number") == COMPARE_PAGE_NUMBER
    and doc.metadata.get("section_title") == COMPARE_SECTION
)

print("===== 简单预处理数据块 =====")
print(simple_example.page_content)
print("metadata:", simple_example.metadata)
print("\n===== 优化预处理数据块 =====")
print(optimized_example.page_content)
print("metadata:", {
    key: optimized_example.metadata.get(key)
    for key in ["source_name", "page_number", "business_type", "section_title", "chunk_index"]
})
print("\n===== 对比结论 =====")
print("简单版本字符数：", len(simple_example.page_content))
print("优化版本字符数：", len(optimized_example.page_content))
print("优化版本章节标题：", optimized_example.metadata["section_title"])

===== 简单预处理数据块 =====
7 可靠性设计 容错能力 ZXVMAX-S通过对关键软件资源的定时检测、实时任务监控、存储保护、数据校验、操作日志 信息保存等手段，可有效地防止小软件故障对系统所造成的冲击，提高了软件系统的容错能力 （即软件错误情况下的自愈能力）。 故障监视及处理 ZXVMAX-S具备自动检测与诊断系统软硬件故障的功能，可对故障硬件实施自动隔离、倒换、 重新启动、重新加载等处理。 满足漏洞扫描 为了避免存在安全漏洞使在网设备受外部攻击，需要使系统满足主流安全漏洞扫描工具的要求。 ZXVMAX-S满足Nessus和CD工具安全漏洞扫描的要求。 7.3 数据可靠性 ZXVMAX-S产品在不同分层架构下的数据都可以确保数据可靠和可恢复。  采集层 采集层运行环境部署采用冗余磁盘阵列设计，在服务器损坏、单一硬盘故障等场景下，通过 替换故障硬件后即可工作，数据不会丢失。同时，系统支持数据补充采集，确保系统数据连 续性。  存储共享层 存储共享层的MPP数据库，根据需要都设置了多个冗余备份，单一节点的损坏不会影响整系 统的运行，也不会导致数据丢失。在替换损坏硬件后，系统能自动进行数据冗余备份的补全 和负载均衡。
metadata: {'producer': 'Apache FOP Version 2.6', 'creator': 'DITA Open Toolkit', 'creationdate': '2023-05-23T21:45:33+08:00', 'source': 'knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', 'file_path': 'knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', 'total_pages': 29, 'format': 'PDF 1.4', 'title': '目录', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20230523214533+08'00'", 'page': 20, 'page_number':

## 批量数据预处理

批量处理文件夹中所有文件

In [21]:
import os

# 获取folder_path下所有文件路径，储存在file_paths里
file_paths = []
folder_path = 'knowledge_path/VMAX-S'
for root, dirs, files in os.walk(folder_path):
    for file in files:
        file_path = os.path.join(root, file)
        file_paths.append(file_path)
print(file_paths)

from langchain_community.document_loaders import PyMuPDFLoader
# from langchain_community.document_loaders import UnstructuredMarkdownLoader

# 遍历文件路径并把实例化的loader存放在loaders里
loaders = []

for file_path in file_paths:
    file_type = file_path.split('.')[-1]
    if file_type == 'pdf':
        loaders.append(PyMuPDFLoader(file_path))
    else:
        print(f"Unsupported file type: {file_type} for file {file_path}")


# 下载文件并存储到text
# 加载所有文档内容到 texts
texts = []
for loader in loaders:
    texts.extend(loader.load())  # 关键步骤：初始化 texts


# 统一复用前面定义的清洗函数，避免单文档与批量处理规则不一致
raw_texts = texts
texts = [clean_pdf_document(doc) for doc in raw_texts if doc.page_content.strip()]
print(f"原始页面数：{len(raw_texts)}，清洗后有效页面数：{len(texts)}")

['knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', 'knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（语音业务）.pdf', 'knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（数据业务）.pdf', 'knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（端到端业务）.pdf', 'knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf']
原始页面数：188，清洗后有效页面数：188


In [22]:
''' 
* RecursiveCharacterTextSplitter 递归字符文本分割
RecursiveCharacterTextSplitter 将按不同的字符递归地分割(按照这个优先级["\n\n", "\n", " ", ""])，
    这样就能尽量把所有和语义相关的内容尽可能长时间地保留在同一位置
RecursiveCharacterTextSplitter需要关注的是4个参数：

* separators - 分隔符字符串数组
* chunk_size - 每个文档的字符数量限制
* chunk_overlap - 两份文档重叠区域的长度
* length_function - 长度计算函数
'''
#导入文本分割器
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [23]:
# 知识库中单段文本长度
CHUNK_SIZE = 512

# 知识库中相邻文本重合长度
OVERLAP_SIZE = 50

In [24]:
# 使用递归字符文本分割器
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=OVERLAP_SIZE,
    separators=["\n\n", "\n", "。", "；", "！", "？", "，", ""],
)
# text_splitter.split_text(text.page_content[0:1000])

In [25]:
split_docs = split_documents_by_section(texts)
print(f"切分后的文件数量：{len(split_docs)}")

切分后的文件数量：451


In [26]:
print(f"切分后的字符数（可以用来大致评估 token 数）：{sum([len(doc.page_content) for doc in split_docs])}")

切分后的字符数（可以用来大致评估 token 数）：101457


In [27]:
split_docs[300].page_content

'2.1 组网结构\n图2-1 ZXVMAX-S端到端系统组网结构图表2-1 ZXVMAX-S端到端系统组件组网说明产品级组件产品（组件）说明应用分析产品实现用户分析、网络分析、专题分析、评估优化、终端分析等业务功能，并能针对Feature独立发布，基于服务器+客户端方式部署。\n存储共享产品\n基于Hadoop软件的服务器集群，实现海量数据的存储、预处理、共享。\n采集层产品-IDS\n面向核心网数据业务抓包，信令抓包，通常需要在各标准接口进行端口映射或分光。'

# 向量数据库的构建

### 安装Milvus数据库

Milvus Standalone 是单机服务器部署。Milvus Standalone 的所有组件都打包到一个[Docker 镜像](https://milvus.io/docs/install_standalone-docker.md)中，部署起来非常方便。如果你有生产工作负载，但又不想使用 Kubernetes，那么在内存充足的单机上运行 Milvus Standalone 是一个不错的选择。此外，Milvus Standalone 通过主从复制支持高可用性。

**使用 Docker Compose 运行 Milvus (Linux)**

Milvus 在 Milvus 资源库中提供了 Docker Compose 配置文件。要使用 Docker Compose 安装 Milvus，只需运行

```shell
# Download the configuration file
$ wget https://github.com/milvus-io/milvus/releases/download/v2.6.4/milvus-standalone-docker-compose.yml -O docker-compose.yml


# Start Milvus
$ sudo docker compose -f docker-compose.yml up -d

Creating milvus-etcd  ... done
Creating milvus-minio ... done
Creating milvus-standalone ... done
```

- 如果运行上述命令失败，请检查系统是否安装了 Docker Compose V1。如果是这种情况，建议你根据[本页](https://docs.docker.com/compose/)的说明迁移到 Docker Compose V2。
- 如果您在拉取镜像时遇到任何问题，请通过[community@zilliz.com](mailto:community@zilliz.com)联系我们，并提供有关问题的详细信息，我们将为您提供必要的支持。

启动 Milvus 后、

- 名为milvus-standalone、milvus-minio和milvus-etcd的容器启动。
  - **milvus-etcd**容器不向主机暴露任何端口，并将其数据映射到当前文件夹中的**volumes/etcd**。
  - **milvus-minio**容器使用默认身份验证凭据在本地为端口**9090**和**9091**提供服务，并将其数据映射到当前文件夹中的**volumes/minio**。
  - **Milvus-standalone**容器使用默认设置为本地**19530**端口提供服务，并将其数据映射到当前文件夹中的**volumes/milvus**。

你可以使用以下命令检查容器是否启动并运行：

```shell
$ sudo docker-compose ps

      Name                     Command                  State                            Ports
--------------------------------------------------------------------------------------------------------------------
milvus-etcd         etcd -advertise-client-url ...   Up             2379/tcp, 2380/tcp
milvus-minio        /usr/bin/docker-entrypoint ...   Up (healthy)   9000/tcp
milvus-standalone   /tini -- milvus run standalone   Up             0.0.0.0:19530->19530/tcp, 0.0.0.0:9091->9091/tcp
```

你还可以访问 Milvus WebUI，网址是`http://127.0.0.1:9091/webui/` ，了解有关 Milvus 实例的更多信息。详情请参阅[Milvus WebUI](https://milvus.io/docs/zh/milvus-webui.md)。

**停止和删除 Milvus**

您可以按以下步骤停止和删除此容器

```shell
# Stop Milvus
$ sudo docker compose down

# Delete service data
$ sudo rm -rf volumes
```

**Attu（Milvus 图形用户界面）**

https://github.com/zilliztech/attu

```
<!-- docker run -d --name attu-container -p 9092:3000 -e MILVUS_URL=129.201.70.35:19530 zilliz/attu:v2. -->

docker run -d --name attu-container -p 9092:3000 -e MILVUS_URL=129.201.70.35:19530 zilliz/attu:v3.0.0-beta.6
```

登陆信息 admin@zte.com.cn / ShiYan@12345

**milvus使用指南**

- [数据库](https://milvus.io/docs/zh/manage_databases.md)
- Collections
- Schema 和数据字段
- 插入和删除
- 索引
- 搜索和 Rerankers

> 参考https://milvus.io/docs/zh/manage_databases.md


## 前序数据预处理

In [28]:
#!/usr/bin/env python
# coding: utf-8

# 批量处理文件夹中所有文件
import os

# 获取folder_path下所有文件路径，储存在file_paths里
file_paths = []
folder_path = 'knowledge_path/VMAX-S'
for root, dirs, files in os.walk(folder_path):
    for file in files:
        file_path = os.path.join(root, file)
        file_paths.append(file_path)
print(file_paths)

from langchain_community.document_loaders import PyMuPDFLoader
# from langchain_community.document_loaders import UnstructuredMarkdownLoader

# 遍历文件路径并把实例化的loader存放在loaders里
loaders = []

for file_path in file_paths:
    file_type = file_path.split('.')[-1]
    if file_type == 'pdf':
        loaders.append(PyMuPDFLoader(file_path))
    else:
        print(f"Unsupported file type: {file_type} for file {file_path}")

# 下载文件并存储到text
# 加载所有文档内容到 texts
texts = []
for loader in loaders:
    texts.extend(loader.load())  # 关键步骤：初始化 texts


# 统一复用前面定义的清洗函数
raw_texts = texts
texts = [clean_pdf_document(doc) for doc in raw_texts if doc.page_content.strip()]
print(f"原始页面数：{len(raw_texts)}，清洗后有效页面数：{len(texts)}")




#导入文本分割器
from langchain_text_splitters import RecursiveCharacterTextSplitter
''' 
* RecursiveCharacterTextSplitter 递归字符文本分割
RecursiveCharacterTextSplitter 将按不同的字符递归地分割(按照这个优先级["\n\n", "\n", " ", ""])，
    这样就能尽量把所有和语义相关的内容尽可能长时间地保留在同一位置
RecursiveCharacterTextSplitter需要关注的是4个参数：

* separators - 分隔符字符串数组
* chunk_size - 每个文档的字符数量限制
* chunk_overlap - 两份文档重叠区域的长度
* length_function - 长度计算函数
'''


# 知识库中单段文本长度
CHUNK_SIZE = 512

# 知识库中相邻文本重合长度
OVERLAP_SIZE = 50


# 使用递归字符文本分割器
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=OVERLAP_SIZE,
    separators=["\n\n", "\n", "。", "；", "！", "？", "，", ""],
)


split_docs = split_documents_by_section(texts)
print(f"切分后的文件数量：{len(split_docs)}")



print(f"切分后的字符数（可以用来大致评估 token 数）：{sum([len(doc.page_content) for doc in split_docs])}")



split_docs[300].page_content



['knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', 'knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（语音业务）.pdf', 'knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（数据业务）.pdf', 'knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（端到端业务）.pdf', 'knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf']
原始页面数：188，清洗后有效页面数：188
切分后的文件数量：451
切分后的字符数（可以用来大致评估 token 数）：101457


'2.1 组网结构\n图2-1 ZXVMAX-S端到端系统组网结构图表2-1 ZXVMAX-S端到端系统组件组网说明产品级组件产品（组件）说明应用分析产品实现用户分析、网络分析、专题分析、评估优化、终端分析等业务功能，并能针对Feature独立发布，基于服务器+客户端方式部署。\n存储共享产品\n基于Hadoop软件的服务器集群，实现海量数据的存储、预处理、共享。\n采集层产品-IDS\n面向核心网数据业务抓包，信令抓包，通常需要在各标准接口进行端口映射或分光。'

## embedding模型准备

In [29]:
import sys
print(sys.executable)

import langchain_core
print(langchain_core.__version__)

import langchain_core.messages as m
print(m.__file__)

/Users/will/opt/anaconda3/envs/env_rag/bin/python
1.4.7
/Users/will/opt/anaconda3/envs/env_rag/lib/python3.12/site-packages/langchain_core/messages/__init__.py


In [30]:
from langchain_ollama import OllamaEmbeddings

# 初始化嵌入模型
my_emb = OllamaEmbeddings(
    base_url="http://192.168.1.166:11434",
    model="qwen3-embedding:0.6b",
)


In [31]:
# from langchain_community.embeddings import XinferenceEmbeddings

# # 初始化嵌入模型
# my_emb = XinferenceEmbeddings(
#     server_url="http://129.201.70.35:9997", 
#     model_uid="my_qwen3_embed_0.6b"
# )


## 构建Milvus向量库-简单检索版本


In [32]:
from pymilvus import MilvusClient, connections
from langchain_milvus import Milvus

# Milvus 连接参数。Quickstart 中使用 MilvusClient(uri=...) 连接数据库；这里连接远端 Milvus Standalone。
connection_args = {
    "uri": "http://localhost:19530",
}

# langchain-milvus 初始化已有集合时会同时使用 MilvusClient 和 PyMilvus ORM Collection。
# 先注册同一个 alias，避免 should create connection first。
_milvus_client = MilvusClient(**connection_args)
_milvus_alias = _milvus_client._using
if not connections.has_connection(_milvus_alias):
    connections.connect(alias=_milvus_alias, uri=connection_args["uri"])

# 简单检索集合只包含一个 dense 向量字段，相当于 Milvus Quickstart 中的单 vector 字段语义检索。
simple_collection_name = "ZXVMAXS"
dense_index_param = {
    "metric_type": "COSINE",
    "index_type": "HNSW",
}

batch_size = 30

try:
    total_batches = (len(split_docs) + batch_size - 1) // batch_size
    vectordb = None

    for batch_num in range(total_batches):
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, len(split_docs))
        batch_docs = split_docs[start_idx:end_idx]

        print(f"正在处理第 {batch_num + 1}/{total_batches} 批文档 (文档 {start_idx}-{end_idx - 1})")

        if batch_num == 0:
            vectordb = Milvus.from_documents(
                documents=batch_docs,
                embedding=my_emb,
                collection_name=simple_collection_name,
                connection_args=connection_args,
                vector_field="dense",
                text_field="text",
                index_params=dense_index_param,
                drop_old=True,
            )
        else:
            vectordb.add_documents(batch_docs)

        print(f"第 {batch_num + 1} 批文档已成功导入 Milvus 简单检索集合")

    # 搜索时只需要返回文本和元数据字段，不需要返回 dense 向量原始值。
    vectordb.fields = [field for field in vectordb.fields if field not in {"dense", "sparse", "vector"}]
    print(f"所有文档已导入简单检索集合：{simple_collection_name}")
    print(f"向量字段：{vectordb.vector_fields}")
    print(f"返回字段：{vectordb.fields}")

except Exception as e:
    print(f"构建 Milvus 简单检索集合时发生错误: {e}")


正在处理第 1/16 批文档 (文档 0-29)
第 1 批文档已成功导入 Milvus 简单检索集合
正在处理第 2/16 批文档 (文档 30-59)
第 2 批文档已成功导入 Milvus 简单检索集合
正在处理第 3/16 批文档 (文档 60-89)
第 3 批文档已成功导入 Milvus 简单检索集合
正在处理第 4/16 批文档 (文档 90-119)
第 4 批文档已成功导入 Milvus 简单检索集合
正在处理第 5/16 批文档 (文档 120-149)
第 5 批文档已成功导入 Milvus 简单检索集合
正在处理第 6/16 批文档 (文档 150-179)
第 6 批文档已成功导入 Milvus 简单检索集合
正在处理第 7/16 批文档 (文档 180-209)
第 7 批文档已成功导入 Milvus 简单检索集合
正在处理第 8/16 批文档 (文档 210-239)
第 8 批文档已成功导入 Milvus 简单检索集合
正在处理第 9/16 批文档 (文档 240-269)
第 9 批文档已成功导入 Milvus 简单检索集合
正在处理第 10/16 批文档 (文档 270-299)
第 10 批文档已成功导入 Milvus 简单检索集合
正在处理第 11/16 批文档 (文档 300-329)
第 11 批文档已成功导入 Milvus 简单检索集合
正在处理第 12/16 批文档 (文档 330-359)
第 12 批文档已成功导入 Milvus 简单检索集合
正在处理第 13/16 批文档 (文档 360-389)
第 13 批文档已成功导入 Milvus 简单检索集合
正在处理第 14/16 批文档 (文档 390-419)
第 14 批文档已成功导入 Milvus 简单检索集合
正在处理第 15/16 批文档 (文档 420-449)
第 15 批文档已成功导入 Milvus 简单检索集合
正在处理第 16/16 批文档 (文档 450-450)
第 16 批文档已成功导入 Milvus 简单检索集合
所有文档已导入简单检索集合：ZXVMAXS
向量字段：['dense']
返回字段：['text', 'pk', 'producer', 'creator', 'creationdate', 'source', 'file_pa

## 构建Milvus向量库-混合检索版本

参考 [Milvus Quickstart](https://milvus.io/docs/v2.6.x/quickstart.md) 的基本流程：连接 Milvus、创建 collection、写入向量和标量字段、搜索时只返回需要展示的文本与元数据字段。这里在 LangChain 封装上对应为：

- `dense`：由 embedding 模型生成，用于语义相似度检索；
- `sparse`：由 Milvus 服务端 BM25 内置函数根据原始文本生成，用于关键词检索；
- 检索时通过 `ranker_type="weighted"` 或 `ranker_type="rrf"` 融合 dense 与 sparse 两路召回结果；
- `sparse` 是 BM25 函数输出的稀疏向量字段，不能作为 raw data 返回，因此检索前要从 `fields` 中排除 `dense/sparse/vector`。

`BM25BuiltInFunction` 不需要客户端维护语料统计，文本分析、稀疏向量生成和 BM25 评分都在 Milvus 服务端完成。注意：Milvus Lite 暂不支持该内置全文检索能力，需要使用 Milvus Standalone、Distributed 或兼容服务。


In [33]:
from pymilvus import MilvusClient, connections
from langchain_milvus import Milvus, BM25BuiltInFunction

connection_args = {
    "uri": "http://localhost:19530",
}

_milvus_client = MilvusClient(**connection_args)
_milvus_alias = _milvus_client._using
if not connections.has_connection(_milvus_alias):
    connections.connect(alias=_milvus_alias, uri=connection_args["uri"])

# 混合检索集合使用独立名称，避免和简单检索集合的单向量字段 schema 冲突。
hybrid_collection_name = "ZXVMAXS_HYBRID"

dense_index_param = {
    "metric_type": "COSINE",
    "index_type": "HNSW",
}
sparse_index_param = {
    "metric_type": "BM25",
    "index_type": "AUTOINDEX",
}

batch_size = 30

try:
    total_batches = (len(split_docs) + batch_size - 1) // batch_size
    vectordb_hybrid = None

    for batch_num in range(total_batches):
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, len(split_docs))
        batch_docs = split_docs[start_idx:end_idx]

        print(f"正在处理第 {batch_num + 1}/{total_batches} 批文档 (文档 {start_idx}-{end_idx - 1})")

        if batch_num == 0:
            vectordb_hybrid = Milvus.from_documents(
                documents=batch_docs,
                embedding=my_emb,
                builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
                vector_field=["dense", "sparse"],
                collection_name=hybrid_collection_name,
                connection_args=connection_args,
                consistency_level="Bounded",
                index_params=[dense_index_param, sparse_index_param],
                drop_old=True,
            )
        else:
            vectordb_hybrid.add_documents(batch_docs)

        print(f"第 {batch_num + 1} 批文档已成功导入 Milvus 混合检索集合")

    # 搜索结果只返回文本和元数据字段；Milvus 不允许返回 BM25 函数输出的 sparse raw data。
    vectordb_hybrid.fields = [
        field for field in vectordb_hybrid.fields
        if field not in {"dense", "sparse", "vector"}
    ]
    print(f"所有文档已导入混合检索集合：{hybrid_collection_name}")
    print(f"向量字段：{vectordb_hybrid.vector_fields}")
    print(f"返回字段：{vectordb_hybrid.fields}")

except Exception as e:
    print(f"构建 Milvus 混合检索集合时发生错误: {e}")


正在处理第 1/16 批文档 (文档 0-29)
第 1 批文档已成功导入 Milvus 混合检索集合
正在处理第 2/16 批文档 (文档 30-59)
第 2 批文档已成功导入 Milvus 混合检索集合
正在处理第 3/16 批文档 (文档 60-89)
第 3 批文档已成功导入 Milvus 混合检索集合
正在处理第 4/16 批文档 (文档 90-119)
第 4 批文档已成功导入 Milvus 混合检索集合
正在处理第 5/16 批文档 (文档 120-149)
第 5 批文档已成功导入 Milvus 混合检索集合
正在处理第 6/16 批文档 (文档 150-179)
第 6 批文档已成功导入 Milvus 混合检索集合
正在处理第 7/16 批文档 (文档 180-209)
第 7 批文档已成功导入 Milvus 混合检索集合
正在处理第 8/16 批文档 (文档 210-239)
第 8 批文档已成功导入 Milvus 混合检索集合
正在处理第 9/16 批文档 (文档 240-269)
第 9 批文档已成功导入 Milvus 混合检索集合
正在处理第 10/16 批文档 (文档 270-299)
第 10 批文档已成功导入 Milvus 混合检索集合
正在处理第 11/16 批文档 (文档 300-329)
第 11 批文档已成功导入 Milvus 混合检索集合
正在处理第 12/16 批文档 (文档 330-359)
第 12 批文档已成功导入 Milvus 混合检索集合
正在处理第 13/16 批文档 (文档 360-389)
第 13 批文档已成功导入 Milvus 混合检索集合
正在处理第 14/16 批文档 (文档 390-419)
第 14 批文档已成功导入 Milvus 混合检索集合
正在处理第 15/16 批文档 (文档 420-449)
第 15 批文档已成功导入 Milvus 混合检索集合
正在处理第 16/16 批文档 (文档 450-450)
第 16 批文档已成功导入 Milvus 混合检索集合
所有文档已导入混合检索集合：ZXVMAXS_HYBRID
向量字段：['dense', 'sparse']
返回字段：['text', 'pk', 'producer', 'creator', 'creationdate', '

下面用同一个 Milvus 集合做一次快速检索验证。`weights` 的顺序与 `vector_field=["dense", "sparse"]` 对应：第一个权重控制语义向量检索，第二个权重控制 BM25 关键词检索。

In [34]:
query = "ZXVMAX-S上网日志有哪些功能？"

# Milvus 不允许把 sparse/dense 向量字段作为 raw data 返回；混合检索前只保留普通字段。
vectordb_hybrid.fields = [
    field for field in vectordb_hybrid.fields
    if field not in {"dense", "sparse", "vector"}
]

hybrid_docs = vectordb_hybrid.similarity_search(
    query,
    k=5,
    ranker_type="weighted",
    ranker_params={"weights": [0.6, 0.4]},
)

for i, doc in enumerate(hybrid_docs, 1):
    source = doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源"
    page = doc.metadata.get("page", "-")
    print(f"===== 混合检索结果 {i} | source={source} | page={page} =====")
    print(doc.page_content[:500])
    print()


===== 混合检索结果 1 | source=knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=13 =====
4 功能
本章包含如下主题：
- 上网日志保存
- 上网日志查询
- 上网日志批量导入查询
- 日志管理
- 账号管理
- 角色管理
- 资源监控
- 告警管理
- 省级网关对接
- 拨测结果自动比对功能
- NAT日志入库功能
- 北向接口
- 上网日志历史查询
- 云化上网日志XDR查询
以下介绍ZXVMAX-S的主要的功能。

===== 混合检索结果 2 | source=knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=4 =====
1.2 产品定位
ZXVMAX-S系统可以长时间的保存用户的上网日志，并可通过公网IP、目的IP、IMSI、MSISDN、URL等参数查询用户上网日志，满足运营商追溯用户上网行为的需求。

===== 混合检索结果 3 | source=knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=0 =====
ZXVMAX-S
多维价值分析系统
产品描述（上网日志业务）
产品版本：V6.23
中兴通讯股份有限公司
地址：深圳市南山区高新技术产业园科技南路中兴通讯大厦邮编：518057电话：0755-26770800400-8301118800-8301118（座机）
技术支持网站：http://support.zte.com.cn电子邮件：800@zte.com.cn

===== 混合检索结果 4 | source=knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=16 =====
4.11 NAT日志入库功能
ZXVMAX-S支持将NAT日志保存到GP数据库，可通过公网IP或目的IP查询NAT日志。

===== 混合检索结果 5 | source=knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（端到端业务）.pdf | page=6 =====
1 产品定位与特点
图1-2 ZXVMAX-S 端

In [35]:
# 也可以使用 RRF（Reciprocal Rank Fusion）融合两路召回结果，适合不想手动调权重的场景。
# Milvus 不允许把 sparse/dense 向量字段作为 raw data 返回；混合检索前只保留普通字段。
vectordb_hybrid.fields = [
    field for field in vectordb_hybrid.fields
    if field not in {"dense", "sparse", "vector"}
]

rrf_docs = vectordb_hybrid.similarity_search(
    query,
    k=5,
    ranker_type="rrf",
    ranker_params={"k": 100},
)

for i, doc in enumerate(rrf_docs, 1):
    source = doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源"
    page = doc.metadata.get("page", "-")
    print(f"===== RRF混合检索结果 {i} | source={source} | page={page} =====")
    print(doc.page_content[:500])
    print()


===== RRF混合检索结果 1 | source=knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=13 =====
4 功能
本章包含如下主题：
- 上网日志保存
- 上网日志查询
- 上网日志批量导入查询
- 日志管理
- 账号管理
- 角色管理
- 资源监控
- 告警管理
- 省级网关对接
- 拨测结果自动比对功能
- NAT日志入库功能
- 北向接口
- 上网日志历史查询
- 云化上网日志XDR查询
以下介绍ZXVMAX-S的主要的功能。

===== RRF混合检索结果 2 | source=knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（端到端业务）.pdf | page=6 =====
1 产品定位与特点
图1-2 ZXVMAX-S 端到端系统架构
- ZXVMAX-S端到端提供4大主要功能：用户分析、网络分析、专题分析和评估优化。
- 综合网络分析，无线核心网综合解决方案。
ZXVMAX-S端到端系统支持对无线数据、核心网数据进行关联，支持问题定界，可以更加准确的定位问题点，确定网络的性能瓶颈，更有针对性的指导优化。
- 多样化处理模式，实时分析、周期任务与指定任务驱动，满足不同用户的需求。
ZXVMAX-S端到端系统支持对网络性能KPI、端到端质量等进行实时分析，仪表盘呈现，对超出门限指标进行预警，指导运维人员快速响应解决问题。
ZXVMAX-S端到端系统支持定制周期任务，利用空闲时段完成运算，让用户第一时间了解关注信息。
ZXVMAX-S端到端系统支持根据用户需求进行数据自定义的查询或分析，满足不同场景下用户分析解决问题的需求。
- 信息呈现与问题定位并存，满足不同层次用户的需求。
ZXVMAX-S端到端系统支持事件详细信息的呈现，也支持基于经验算法的问题定位，提升分析解决问题的效率，满足不同层次用户的需求。
- 多种应用，自由选择。

===== RRF混合检索结果 3 | source=knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=4 =====
1.2 产品定位
ZXVMAX-S系统可以长时间的保存用户的上网日志，并可通过公网IP、目的IP、IMSI、MS

## 连接数据库进行测试


In [37]:
from langchain_milvus import Milvus
from pymilvus import connections, utility

def list_all_collections():
    """获取所有collection列表"""
    try:
        # 连接到Milvus（使用与vectordb相同的参数）
        connections.connect(
            alias="default",
            host="localhost",
            port="19530"
        )
        
        # 获取所有collection
        collections = utility.list_collections()
        print("所有Collection列表:")
        for i, coll_name in enumerate(collections, 1):
            print(f"{i}. {coll_name}")
            
        return collections
        
    except Exception as e:
        print("获取collection列表失败:", e)
        return []
    finally:
        # 断开连接
        connections.disconnect("default")

# 执行
collections = list_all_collections()
print(f"总共找到 {len(collections)} 个collection")

所有Collection列表:
1. ZXVMAXS
2. ZXVMAXS_HYBRID
总共找到 2 个collection


## 简单检索版本与混合检索版本结果对比

参考 [Milvus Quickstart](https://milvus.io/docs/v2.6.x/quickstart.md) 中的检索流程：先把文档向量和元数据写入 collection，再用同一个 query 发起 `search` 并查看 Top-K 命中的文本与元数据。这里沿用同一思路，对比前面创建的两个 Milvus collection：

- **简单检索版本 `ZXVMAXS`**：只使用 `dense` 向量字段，主要依赖语义相似度召回；
- **混合检索版本 `ZXVMAXS_HYBRID`**：同时使用 `dense` 语义向量和 `sparse` BM25 关键词向量，再通过 weighted ranker 融合结果。

如果 query 中包含产品名、告警码、功能名等关键词，混合检索通常更容易把关键词精确命中的片段排到前面；如果 query 更偏自然语言语义表达，简单向量检索和混合检索的结果可能会有较多重合。


In [39]:
from pymilvus import MilvusClient, connections
from langchain_milvus import Milvus, BM25BuiltInFunction
from langchain_ollama import OllamaEmbeddings

my_emb = OllamaEmbeddings(
    base_url="http://192.168.1.166:11434",
    model="qwen3-embedding:0.6b",
)

connection_args = {
    "uri": "http://localhost:19530",
}

# langchain-milvus 0.3.x 初始化已有集合时，会同时使用 MilvusClient 和旧版 PyMilvus ORM Collection。
# MilvusClient 会生成一个 cm-* alias；这里提前把同一个 alias 注册给 ORM，避免 should create connection first。
_milvus_client = MilvusClient(**connection_args)
_milvus_alias = _milvus_client._using
if not connections.has_connection(_milvus_alias):
    connections.connect(alias=_milvus_alias, uri=connection_args["uri"])

vectordb = Milvus(
    embedding_function=my_emb,
    collection_name="ZXVMAXS",  # 简单检索集合：只有 dense 向量字段
    connection_args=connection_args,
    vector_field="dense",
    text_field="text",
)

vectordb_hybrid = Milvus(
    embedding_function=my_emb,
    builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
    collection_name="ZXVMAXS_HYBRID",  # 混合检索集合：dense + sparse 两个向量字段
    connection_args=connection_args,
    vector_field=["dense", "sparse"],
    text_field="text",
    consistency_level="Bounded",
)

# Milvus 不允许把向量字段作为 raw data 返回，尤其是 BM25 生成的 sparse 字段。
# 这里显式删除 dense/sparse/vector，比只依赖 vector_fields 更稳。
for vectorstore in [vectordb, vectordb_hybrid]:
    vectorstore.fields = [
        field for field in vectorstore.fields
        if field not in {"dense", "sparse", "vector"}
    ]


In [40]:
comparison_query = "ZXVMAX-S上网日志有哪些功能？"
k = 5

# Milvus 不允许把 sparse/dense 向量字段作为 raw data 返回；检索前只保留普通字段。
for vectorstore in [vectordb, vectordb_hybrid]:
    vectorstore.fields = [
        field for field in vectorstore.fields
        if field not in {"dense", "sparse", "vector"}
    ]

print("简单检索返回字段：", vectordb.fields)
print("混合检索返回字段：", vectordb_hybrid.fields)

simple_docs = vectordb.similarity_search(comparison_query, k=k)
hybrid_docs = vectordb_hybrid.similarity_search(
    comparison_query,
    k=k,
    ranker_type="weighted",
    ranker_params={"weights": [0.6, 0.4]},  # dense 权重 0.6，BM25 sparse 权重 0.4
)

def doc_key(doc):
    source = str(doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源").strip()
    page = str(doc.metadata.get("page", "-")).strip()
    return source, page

def print_results(title, docs):
    print(f"\n===== {title} =====")
    displayed = set()
    row = 1
    for doc in docs:
        source = str(doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源").strip()
        page = str(doc.metadata.get("page", "-")).strip()
        display_key = (source, page)
        if display_key in displayed:
            continue
        displayed.add(display_key)
        preview = doc.page_content[:180].replace("\n", " ")
        print(f"{row}. source={source} | page={page}")
        print(f"   {preview}")
        row += 1

print(f"Query: {comparison_query}")
print_results("简单检索 Top-K（dense semantic search）", simple_docs)
print_results("混合检索 Top-K（dense + BM25 sparse weighted fusion）", hybrid_docs)

simple_keys = {doc_key(doc): i + 1 for i, doc in enumerate(simple_docs)}
hybrid_keys = {doc_key(doc): i + 1 for i, doc in enumerate(hybrid_docs)}
common_keys = set(simple_keys) & set(hybrid_keys)
only_simple = set(simple_keys) - set(hybrid_keys)
only_hybrid = set(hybrid_keys) - set(simple_keys)

print("\n===== Top-K 差异统计 =====")
print(f"共同召回数量：{len(common_keys)}/{k}")
print(f"仅简单检索命中：{len(only_simple)}")
print(f"仅混合检索命中：{len(only_hybrid)}")

if only_hybrid:
    print("\n混合检索独有结果示例：")
    hybrid_doc_by_key = {doc_key(doc): doc for doc in hybrid_docs}
    for key in list(only_hybrid)[:3]:
        doc = hybrid_doc_by_key[key]
        source, page = key
        preview = doc.page_content[:120].replace("\n", " ")
        print(f"- source={source} | page={page} | {preview}")

print("\n对比结论：")
print("- 简单检索只看 dense 向量语义相似度，适合语义改写较多的问题。")
print("- 混合检索同时融合 dense 语义召回和 BM25 关键词召回，更适合包含产品名、模块名、告警码、专有名词的问题。")
print("- 如果共同召回数量较低，说明关键词通道显著改变了排序或召回集合；如果共同召回数量较高，说明该 query 的语义召回已经比较稳定。")


简单检索返回字段： ['text', 'pk', 'producer', 'creator', 'creationdate', 'source', 'file_path', 'total_pages', 'format', 'title', 'author', 'subject', 'keywords', 'moddate', 'trapped', 'modDate', 'creationDate', 'page', 'source_name', 'page_number', 'cleaned', 'business_type', 'section_title', 'chunk_index']
混合检索返回字段： ['text', 'pk', 'producer', 'creator', 'creationdate', 'source', 'file_path', 'total_pages', 'format', 'title', 'author', 'subject', 'keywords', 'moddate', 'trapped', 'modDate', 'creationDate', 'page', 'source_name', 'page_number', 'cleaned', 'business_type', 'section_title', 'chunk_index']
Query: ZXVMAX-S上网日志有哪些功能？

===== 简单检索 Top-K（dense semantic search） =====
1. source=knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=13
   4 功能 本章包含如下主题： - 上网日志保存 - 上网日志查询 - 上网日志批量导入查询 - 日志管理 - 账号管理 - 角色管理 - 资源监控 - 告警管理 - 省级网关对接 - 拨测结果自动比对功能 - NAT日志入库功能 - 北向接口 - 上网日志历史查询 - 云化上网日志XDR查询 以下介绍ZXVMAX-S的主要的功能。
2. source=knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=4
   1.2 产

### 对比结果解读

本次真实环境中，`ZXVMAXS` 和 `ZXVMAXS_HYBRID` 都使用同一批 `split_docs` 构建，因此两者的 Top-K 结果有较高重合是正常现象。查询 `ZXVMAX-S上网日志有哪些功能？` 同时包含产品名和功能类关键词，`dense` 语义检索已经能稳定召回“上网日志业务”相关页面，所以简单检索和混合检索共同命中 4 条。

混合检索相比简单检索额外命中了“端到端业务”文档中的功能概述页，说明 BM25 关键词通道会强化 `ZXVMAX-S`、`功能` 等字面匹配，再与 dense 语义召回结果融合排序。这个结果可以说明两者差异：

- 简单检索更依赖语义相似度，适合自然语言改写后的问题；
- 混合检索同时考虑语义相似度和关键词命中，适合包含产品名、模块名、告警码、接口名等专有词的问题；
- 检索时只返回 `text/source/page` 等普通字段，不返回 `dense/sparse` 向量字段，符合 Milvus Quickstart 中通过 `output_fields` 控制返回字段的做法，也避免 `not allowed to retrieve raw data of field sparse` 错误。


# 构建检索问答链-基础版

本章基于上一章构建好的 **简单检索向量库 `ZXVMAXS`** 搭建 RAG 问答链。流程可以拆成四步：

1. 连接 Milvus，加载只包含 `dense` 向量字段的简单检索 collection；
2. 将用户问题转成向量，从 Milvus 中召回 Top-K 相关文档片段；
3. 把召回片段拼接进 Prompt，约束大模型只基于上下文回答；
4. 对比“有知识库”和“无知识库”回答，观察 RAG 对专业问题和幻觉抑制的作用。

这一章暂不使用 BM25 sparse 混合检索，目的是先把最基础的 dense 向量检索问答链讲清楚。


## 加载向量数据库 Milvus

这里加载上一章创建的简单检索集合 `ZXVMAXS`。需要注意三点：

- `embedding_function` 必须和构建向量库时使用的 embedding 模型一致；
- collection 的向量字段名是 `dense`，文本字段名是 `text`，不能使用 LangChain 默认的 `vector`；
- 检索结果只需要返回文本和元数据字段，不需要返回原始向量字段，所以后面会从 `fields` 中排除 `dense/sparse/vector`。


In [42]:
from pymilvus import MilvusClient, connections
from langchain_milvus import Milvus
from langchain_ollama import OllamaEmbeddings

my_emb = OllamaEmbeddings(
    base_url="http://192.168.1.166:11434",
    model="qwen3-embedding:0.6b",
)

connection_args = {
    "uri": "http://localhost:19530",
}

# langchain-milvus 初始化已有集合时会同时使用 MilvusClient 和 PyMilvus ORM Collection。
# 先注册同一个 alias，避免 should create connection first。
_milvus_client = MilvusClient(**connection_args)
_milvus_alias = _milvus_client._using
if not connections.has_connection(_milvus_alias):
    connections.connect(alias=_milvus_alias, uri=connection_args["uri"])

vectordb = Milvus(
    embedding_function=my_emb,
    collection_name="ZXVMAXS",
    connection_args=connection_args,
    vector_field="dense",
    text_field="text",
)

# 搜索结果只需要返回文本和元数据字段，不返回 dense 向量原始值。
vectordb.fields = [
    field for field in vectordb.fields
    if field not in {"dense", "sparse", "vector"}
]

print("向量字段：", vectordb.vector_fields)
print("返回字段：", vectordb.fields)


向量字段： ['dense']
返回字段： ['text', 'pk', 'producer', 'creator', 'creationdate', 'source', 'file_path', 'total_pages', 'format', 'title', 'author', 'subject', 'keywords', 'moddate', 'trapped', 'modDate', 'creationDate', 'page', 'source_name', 'page_number', 'cleaned', 'business_type', 'section_title', 'chunk_index']


In [43]:
results = vectordb.similarity_search(query="什么是vmax的上网日志系统？", k=2)

for i, doc in enumerate(results, 1):
    source = doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源"
    page = doc.metadata.get("page", "-")
    preview = doc.page_content[:220].replace("\n", " ")
    print(f"===== 召回结果 {i} | source={source} | page={page} =====")
    print(preview)
    print()


===== 召回结果 1 | source=knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=4 =====
1.2 产品定位 ZXVMAX-S系统可以长时间的保存用户的上网日志，并可通过公网IP、目的IP、IMSI、MSISDN、URL等参数查询用户上网日志，满足运营商追溯用户上网行为的需求。

===== 召回结果 2 | source=knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=13 =====
4 功能 本章包含如下主题： - 上网日志保存 - 上网日志查询 - 上网日志批量导入查询 - 日志管理 - 账号管理 - 角色管理 - 资源监控 - 告警管理 - 省级网关对接 - 拨测结果自动比对功能 - NAT日志入库功能 - 北向接口 - 上网日志历史查询 - 云化上网日志XDR查询 以下介绍ZXVMAX-S的主要的功能。



## 创建一个 LLM

这里使用 SiliconFlow 的 OpenAI-compatible API 初始化一个聊天模型。教学时可以把它理解为“生成器”：检索器负责从知识库找上下文，LLM 负责根据上下文组织最终答案。

运行前需要确认仓库根目录 `.env` 中已配置 `SILICONFLOW_API_KEY`。


**siliconflow版本为例**

In [49]:
import os
import httpx
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 加载环境变量
env_path = "/Users/will/workspace/llm-rag/.env"
load_dotenv(env_path)
# api_key = os.getenv("SILICONFLOW_API_KEY")
api_key = "<your-siliconflow-key>"
if not api_key:
    raise ValueError("未找到 SILICONFLOW_API_KEY，请先在仓库根目录 .env 中配置。")

# 初始化大语言模型
my_llm = init_chat_model(
    model="Qwen/Qwen3-8B",
    model_provider="openai",
    base_url="https://api.siliconflow.cn/v1",
    api_key=api_key,
    http_client=httpx.Client(trust_env=False),
    temperature=0.3,
    max_tokens=3000,
)

result = my_llm.invoke("你是谁？/no_think")
print(result.content)




我是通义千问，由通义实验室研发的超大规模语言模型。我能够回答问题、创作文字，并进行多种语言的理解和生成。我的训练数据截止到2024年，因此我所知道的信息都来自于此之前。如果你有任何问题或需要帮助，我会尽力提供支持。


## 构建检索问答链

Prompt 是 RAG 问答链的关键控制点。这里的模板强调三件事：

- 只能基于召回上下文回答，减少脱离知识库的发挥；
- 如果上下文没有答案，明确回答“我不知道”；
- 输出保持简洁、连贯，适合教学演示。


In [50]:
from langchain_core.prompts import PromptTemplate

template = """你是 VMAX 运维助手，请严格根据以下上下文信息回答问题。

上下文：
{context}

问题：{question}

回答要求：
1. 只基于上下文回答，不要编造上下文之外的信息；
2. 如果上下文没有提供答案，请回答“我不知道”；
3. 回答要简洁、连贯，不要重复相同信息；
4. 最后说“谢谢你的提问！”。

回答："""

QA_CHAIN_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=template,
)


#### 创建一个基于模板的检索链： 基础检索版本

In [51]:
from langchain_classic.chains import RetrievalQA

# 基础版先使用普通相似度检索，便于理解：问题向量 -> Milvus Top-K 文档片段。
# 如果希望增加结果多样性，可以把 search_type 改为 "mmr"，并设置 fetch_k/lambda_mult。
base_retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)


In [52]:
# 创建问答链
qa_chain = RetrievalQA.from_chain_type(
    my_llm,
    retriever=base_retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)

#### 创建一个基于模板的检索链：LCEL 版本


LCEL（LangChain Expression Language）可以把“检索文档 → 格式化上下文 → 填充 Prompt → 调用 LLM → 解析输出”串成一条可组合的链。

In [53]:
def format_docs(docs):
    """把召回文档拼成 prompt 中的 context，并保留简单来源标记，便于观察依据。"""
    formatted = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源"
        page = doc.metadata.get("page", "-")
        formatted.append(f"[文档{i} | source={source} | page={page}]\n{doc.page_content}")
    return "\n\n".join(formatted)


In [54]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": base_retriever | format_docs, "question": RunnablePassthrough()}
    | QA_CHAIN_PROMPT
    | my_llm
    | StrOutputParser()
)

In [55]:
question_1 = "什么是ZXVMAX-S的5GC业务？"
result = rag_chain.invoke(question_1)
print("LCEL检索链回答 question_1 的结果：")
print(result)

LCEL检索链回答 question_1 的结果：


ZXVMAX-S的5GC业务是指该系统支持多种应用，如信令回溯、记录查询、呼叫分析、网络质量分析、业务质量定界和时延分析等，可根据目标用户需求开启对应权限并部署相应模块。  

谢谢你的提问！


## 检索问答链效果测试

下面用两个问题测试基础版 RAG 链：

- `question_1` 是知识库内的专业问题，用来观察 RAG 是否能从 VMAX 文档中召回依据并生成答案；
- `question_2` 是知识库外的常识问题，用来观察 Prompt 是否能约束模型在缺少上下文时回答“我不知道”。


### 知识库内问题：专业问题回答

这个问题会经过“问题向量化 → Milvus 相似度检索 → 召回片段填入 Prompt → LLM 生成答案”的完整 RAG 流程。代码同时打印前 3 个来源文档，帮助学习者理解答案依据来自哪里。


In [56]:
question_1 = "什么是ZXVMAX-S的5GC业务？/no_think"
result = qa_chain.invoke({"query": question_1})

print("大模型+知识库后回答 question_1 的结果：")
print(result["result"])
print("\n来源文档数量：", len(result["source_documents"]))
for i, doc in enumerate(result["source_documents"][:3], 1):
    source = doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源"
    page = doc.metadata.get("page", "-")
    print(f"来源 {i}: source={source} | page={page}")


大模型+知识库后回答 question_1 的结果：


ZXVMAX-S的5GC业务是指多维价值分析系统在5G核心网（5GC）场景下的应用，支持信令回溯、记录查询、呼叫分析、网络质量分析、业务质量定界、时延分析等多种功能，可根据用户需求部署相应模块。谢谢你的提问！

来源文档数量： 5
来源 1: source=knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=0
来源 2: source=knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（端到端业务）.pdf | page=6
来源 3: source=knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（语音业务）.pdf | page=0


### 知识库外问题：拒答能力测试

这个问题与 VMAX 知识库无关。理想情况下，检索到的上下文无法支持回答，Prompt 会约束模型输出“我不知道”，而不是根据通用知识自由发挥。


In [57]:
question_2 = "周杰伦是谁？/no_think"
result = qa_chain.invoke({"query": question_2})

print("大模型+知识库后回答 question_2 的结果：")
print(result["result"])


大模型+知识库后回答 question_2 的结果：


我不知道。谢谢你的提问！


### 无知识库大模型回答对比

为了体现 RAG 的作用，下面绕过向量数据库，直接让大模型回答同样的问题。对比时重点观察：

- 专业问题是否容易出现泛化、不准确或脱离文档的回答；
- 知识库外问题是否会被直接回答，而不是拒答。


In [58]:
prompt_template = f"请回答下列问题:\n{question_1}"

raw_result_1 = my_llm.invoke(prompt_template)
print("无知识库回答 question_1 的结果：")
print(raw_result_1.content)


无知识库回答 question_1 的结果：


ZXVMAX-S 是华为推出的一款 5G 核心网（5GC）业务平台，主要用于提供 5G 网络中的核心网功能，支持多种 5G 业务场景。它是一个基于云原生架构的 5GC 解决方案，旨在满足运营商在 5G 时代对网络灵活性、可扩展性、高可靠性和低延迟等特性的需求。

### ZXVMAX-S 的 5GC 业务主要包括以下几个方面：

1. **5G 核心网功能模块**：
   - **接入和移动性管理功能（AMF）**：负责用户接入和移动性管理，包括注册、连接管理、移动性控制等。
   - **会话管理功能（SMF）**：负责会话建立、修改和释放，管理用户数据的路由和转发。
   - **用户面功能（UPF）**：处理用户数据的转发和路由，支持网络切片、本地数据分流等功能。
   - **认证授权功能（AUSF）**：负责用户身份认证和授权，确保网络安全性。
   - **策略控制功能（PCF）**：提供策略控制规则，用于网络策略的制定和管理。
   - **网络切片管理功能（NSSF）**：管理不同网络切片的配置和选择，支持多场景业务需求。
   - **统一数据管理（UDM）**：存储用户数据，如用户订阅信息、鉴权数据等。
   - **策略控制器（PCRF）**：提供策略决策功能，用于流量控制和计费策略。

2. **支持多种 5G 业务场景**：
   - **eMBB（增强移动宽带）**：如高清视频、VR/AR、4K/8K 视频等。
   - **URLLC（超可靠低时延通信）**：如工业自动化、远程手术、自动驾驶等。
   - **mMTC（大规模机器类通信）**：如物联网、智能电表、环境监测等。

3. **云原生架构**：
   - ZXVMAX-S 采用微服务架构，支持容器化部署，具备良好的可扩展性和弹性，能够快速响应业务变化。
   - 支持 NFV（网络功能虚拟化）和 SDN（软件定义网络），实现网络功能的灵活部署和管理。

4. **高可靠性和低延迟**：
   - 通过分布式部署和负载均衡技术，确保核心网的高可用性和低延迟。
   - 支持多实例冗余部署，提升系统可靠性。

5. **开放接口和生态兼容**：
   - 提供标准化的开放接口（如 OMAPI、RESTful API），便于与第三方

In [59]:
prompt_template = f"请回答下列问题:\n{question_2}"

raw_result_2 = my_llm.invoke(prompt_template)
print("无知识库回答 question_2 的结果：")
print(raw_result_2.content)


无知识库回答 question_2 的结果：


周杰伦（Jay Chou），1979年1月18日出生于中国台湾省新北市，是华语乐坛极具影响力的音乐人、歌手、导演、编剧和制作人。他以其独特的音乐风格、才华横溢的创作能力以及广泛的粉丝基础闻名于世。

### 主要成就与特点：
1. **音乐创作**：  
   周杰伦以融合中西方音乐元素著称，擅长将中国风（如古筝、二胡、京剧等传统乐器和元素）与现代流行、R&B、嘻哈等风格结合。他的作品常带有浓厚的个人情感和文化特色，例如《青花瓷》《菊花台》《东风破》等经典歌曲。

2. **代表作品**：  
   - **专辑**：《范特西》《八度空间》《叶惠美》《以父之名》《跨时代》等。  
   - **歌曲**：《七里香》《简单爱》《告白气球》《说好不哭》《本草纲目》等。  
   - **电影**：自导自演的《不能说的秘密》《天台爱情》《青花瓷》等，展现了他在影视领域的才华。

3. **影响力**：  
   他是华语乐坛“周杰伦时代”的代表人物，多次获得金曲奖等重要音乐奖项，并推动了华语音乐的国际化。他的音乐风格和歌词创作也对后辈音乐人产生了深远影响。

4. **其他领域**：  
   除了音乐，他还涉足电影导演、编剧、时尚设计等领域，展现了多方面的艺术天赋。

### 个人背景：
- 周杰伦毕业于台湾国立艺术学院（现国立台湾艺术大学）流行音乐系，早期以音乐制作人身份出道，后转型为歌手。  
- 他以“周董”昵称被粉丝熟知，性格低调但才华横溢，常被形容为“音乐鬼才”。

周杰伦的音乐和作品跨越了年龄与文化界限，成为全球华语音乐爱好者的共同记忆。


### 效果小结

通过以上两个问题可以看到：接入知识库后，模型会优先依据本地文档回答 VMAX 专业问题；当问题与知识库无关时，Prompt 会约束模型回答“我不知道”。不接入知识库时，模型只能依赖自身参数知识，容易给出泛化甚至不准确的回答。这正是 RAG 缓解幻觉、补充私有知识的核心价值。


## 添加历史对话的记忆功能（可选）

基础检索问答链每次只处理当前问题。如果用户提出“继续整理成表格”“上面结果再简化一下”这类依赖上一轮上下文的问题，就需要加入对话历史。

下面使用 `ConversationalRetrievalChain` 演示最简单的多轮问答。注意：这里使用的是 `langchain_classic` 的教学 API，当前版本会有 deprecation warning，但不影响本课程示例运行。


In [60]:
from langchain_classic.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
)


/var/folders/4f/lx3tlt3d6fz2bys8ct46fzxh0000gn/T/ipykernel_855/1922486086.py:3: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(


### 对话检索链（ConversationalRetrievalChain）

对话检索链会先结合历史对话改写或理解当前问题，再进行检索和回答。它适合演示“第二轮问题依赖第一轮答案”的场景。


In [61]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import ConversationalRetrievalChain

template = """你是 VMAX 运维助手，请参考以下对话历史和上下文回答问题。

对话历史：
{chat_history}

相关上下文：
{context}

问题：{question}

回答要求：
1. 只基于上下文回答，不要编造上下文之外的信息；
2. 如果上下文没有提供答案，请回答“我不知道”；
3. 最后说“谢谢你的提问！”。

回答："""

QA_PROMPT = PromptTemplate(
    input_variables=["chat_history", "context", "question"],
    template=template,
)

conversation_qa_chain = ConversationalRetrievalChain.from_llm(
    llm=my_llm,
    retriever=base_retriever,
    memory=memory,
    combine_docs_chain_kwargs={"prompt": QA_PROMPT},
    chain_type="stuff",
)


In [62]:
questions = [
    "什么是ZXVMAX-S的5GC业务？包含哪些功能？/no_think",
    "将以上结果整理成markdown表格形式/no_think",
]

for question in questions:
    result = conversation_qa_chain.invoke({"question": question})
    print(f"问题：{question}")
    print(f"回答：{result['answer']}")
    print("\n" + "=" * 50 + "\n")


问题：什么是ZXVMAX-S的5GC业务？包含哪些功能？/no_think
回答：

ZXVMAX-S的5GC业务是指中兴通讯提供的多维价值分析系统，专门用于支持5G核心网（5GC）相关的运维和分析需求。根据相关上下文，ZXVMAX-S端到端支持以下功能：

- 信令回溯  
- 记录查询  
- 呼叫分析  
- 网络质量分析  
- 业务质量定界  
- 时延分析  

这些功能可以根据目标用户的需求，开启对应权限，部署相应模块，以提升分析和解决问题的效率，满足不同层次用户的需求。  

谢谢你的提问！


问题：将以上结果整理成markdown表格形式/no_think
回答：

| 功能名称               | 功能描述                                                                 |
|------------------------|--------------------------------------------------------------------------|
| 信令回溯               | 支持对5G核心网信令进行回溯分析，帮助定位问题。                           |
| 记录查询               | 提供对5GC相关记录的查询功能，支持多种字段筛选。                             |
| 呼叫分析               | 分析5G核心网中的呼叫行为，识别异常或性能问题。                             |
| 网络质量分析           | 分析网络性能指标，评估网络质量。                                           |
| 业务质量定界           | 定位业务质量问题，帮助进行问题根源分析。                                   |
| 时延分析               | 分析网络中的时延情况，优化网络性能。                                       |
| 查询结果导出           | 支持将查询结果导出为C

### 对话链结果解读

第一轮问题会从 Milvus 中检索 `ZXVMAXS` 简单向量库，生成关于 5GC 业务的回答；第二轮问题没有重复完整背景，只说“以上结果”，对话链会借助 `memory` 中保存的历史问题和回答继续生成表格。这说明 memory 解决的是“多轮上下文连续性”问题，而向量数据库解决的是“外部知识检索”问题，两者职责不同。


# RAG进阶-混合检索

基础版 RAG 只使用 `dense` 语义向量检索。它适合自然语言问题，但对产品名、告警码、接口名、缩写等“字面关键词”不一定稳定。本章使用上一章构建的 `ZXVMAXS_HYBRID` 集合，演示 **dense 语义检索 + sparse BM25 关键词检索** 的混合检索，并进一步加入 Rerank 精排。

本章仍然围绕 VMAX 文档做教学实验，重点观察三件事：

1. 混合检索如何融合语义召回和关键词召回；
2. weighted 与 RRF 两种融合策略的差异；
3. Rerank 如何从召回候选中重新筛选更适合输入 LLM 的上下文。


## 混合检索原理

混合检索通常包含两条召回路径：

- **Dense 向量检索**：将 query 和文档 chunk 编码为 dense embedding，按语义相似度召回，适合自然语言表达、同义改写、模糊问题；
- **Sparse / BM25 检索**：基于关键词、词频和逆文档频率召回，适合产品名、模块名、告警码、接口名、缩写、数字编号等精确匹配。

Milvus 2.6 支持通过 `BM25BuiltInFunction` 在服务端生成 sparse 向量。也就是说，客户端只需要写入 `text` 和 `dense` embedding，Milvus 会根据 `text` 自动生成 `sparse` 字段并参与混合检索。

注意：`sparse` 是 BM25 函数输出的稀疏向量字段，只能用于检索打分，不能作为 raw data 返回。因此使用 LangChain Milvus 检索前，需要从 `vectorstore.fields` 中排除 `dense/sparse/vector`。


## 混合检索实现：先手写一版帮助理解

在直接调用 Milvus 内置混合检索之前，先用一段教学代码把“混合”这件事拆开看清楚：

1. **Dense 路径**：从简单向量库 `ZXVMAXS` 做语义向量检索；
2. **BM25 路径**：从同一批文档文本构建本地 BM25 关键词检索；
3. **分数归一化**：把两路分数都压到 0-1；
4. **加权融合**：`hybrid_score = dense_weight * dense_score + bm25_weight * bm25_score`；
5. **排序输出**：观察不同权重下结果如何变化。

这段代码不是最终工程推荐做法，目的是让学生直观看到混合检索的底层逻辑。理解之后，再使用 Milvus 服务端内置 BM25 和 ranker，会更自然。


### 准备教学版 Dense 与 BM25 两路召回

这里使用 `ZXVMAXS` 简单检索集合中的文本字段构建 BM25。为了课堂运行速度，最多读取 500 条文本；当前示例数据约 322 条，可以完整读取。


In [64]:
from pymilvus import MilvusClient, connections
from langchain_milvus import Milvus
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document
from rank_bm25 import BM25Okapi
import re

my_emb = OllamaEmbeddings(
    base_url="http://192.168.1.166:11434",
    model="qwen3-embedding:0.6b",
)

connection_args = {
    "uri": "http://192.168.1.188:19530",
}

_milvus_client = MilvusClient(**connection_args)
_milvus_alias = _milvus_client._using
if not connections.has_connection(_milvus_alias):
    connections.connect(alias=_milvus_alias, uri=connection_args["uri"])

# Dense 语义检索路径：使用简单检索集合 ZXVMAXS。
vectordb_dense = Milvus(
    embedding_function=my_emb,
    collection_name="ZXVMAXS",
    connection_args=connection_args,
    vector_field="dense",
    text_field="text",
)
vectordb_dense.fields = [field for field in vectordb_dense.fields if field not in {"dense", "sparse", "vector"}]

# BM25 关键词路径：从同一个 collection 读取 text/source/page，构建本地 BM25 索引。
raw_rows = _milvus_client.query(
    collection_name="ZXVMAXS",
    filter="pk >= 0",
    output_fields=["text", "source", "file_path", "page"],
    limit=500,
)

bm25_docs = [
    Document(
        page_content=row["text"],
        metadata={
            "source": row.get("source") or row.get("file_path") or "未知来源",
            "page": row.get("page", "-"),
        },
    )
    for row in raw_rows
]

def tokenize_zh_en(text: str):
    """教学用 tokenizer：英文/数字按词切，中文按 bigram 切，足够演示 BM25。"""
    text = text.lower()
    tokens = re.findall(r"[a-z0-9][a-z0-9_-]+", text)
    chinese = "".join(re.findall(r"[\u4e00-\u9fff]+", text))
    tokens.extend(chinese[i:i + 2] for i in range(max(len(chinese) - 1, 0)))
    return [token for token in tokens if len(token) >= 2]

bm25_corpus = [tokenize_zh_en(doc.page_content) for doc in bm25_docs]
bm25_index = BM25Okapi(bm25_corpus)

print(f"Dense collection: ZXVMAXS")
print(f"BM25 文档数：{len(bm25_docs)}")


MilvusException: <MilvusException: (code=2, message=Fail connecting to server on 192.168.1.188:19530, illegal connection params or server unavailable)>

### 固定权重融合

固定权重适合讲解“混合检索分数如何合成”。下面手动设置 `dense_weight=0.6`、`bm25_weight=0.4`，分别代表语义检索和关键词检索的重要性。


In [ ]:
def normalize_scores(score_map):
    if not score_map:
        return {}
    values = list(score_map.values())
    min_score, max_score = min(values), max(values)
    if max_score == min_score:
        return {key: 1.0 for key in score_map}
    return {key: (value - min_score) / (max_score - min_score) for key, value in score_map.items()}

def doc_key(doc):
    source = str(doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源").strip()
    page = str(doc.metadata.get("page", "-")).strip()
    return source, page, doc.page_content[:80]

def manual_hybrid_search(query, dense_weight=0.6, bm25_weight=0.4, dense_k=8, bm25_k=8, final_k=5):
    # Dense 路径：使用带分数检索。Milvus 返回的分数方向与距离策略有关，教学中做 min-max 归一化后用于相对排序观察。
    dense_pairs = vectordb_dense.similarity_search_with_score(query, k=dense_k)
    dense_docs = [doc for doc, score in dense_pairs]
    dense_raw = {doc_key(doc): float(score) for doc, score in dense_pairs}
    doc_by_key = {doc_key(doc): doc for doc in dense_docs}

    # BM25 路径：本地关键词检索，分数越高越相关。
    tokenized_query = tokenize_zh_en(query)
    bm25_scores = bm25_index.get_scores(tokenized_query)
    top_bm25_indices = sorted(range(len(bm25_scores)), key=lambda idx: bm25_scores[idx], reverse=True)[:bm25_k]
    bm25_raw = {}
    for idx in top_bm25_indices:
        doc = bm25_docs[idx]
        key = doc_key(doc)
        bm25_raw[key] = float(bm25_scores[idx])
        doc_by_key.setdefault(key, doc)

    dense_norm = normalize_scores(dense_raw)
    bm25_norm = normalize_scores(bm25_raw)

    fused = []
    for key, doc in doc_by_key.items():
        dense_score = dense_norm.get(key, 0.0)
        bm25_score = bm25_norm.get(key, 0.0)
        hybrid_score = dense_weight * dense_score + bm25_weight * bm25_score
        fused.append((hybrid_score, dense_score, bm25_score, doc))

    fused.sort(key=lambda item: item[0], reverse=True)
    return fused[:final_k]

def print_manual_hybrid_results(title, results):
    print(f"\n===== {title} =====")
    for i, (hybrid_score, dense_score, bm25_score, doc) in enumerate(results, 1):
        source = doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源"
        page = doc.metadata.get("page", "-")
        preview = doc.page_content[:160].replace("\n", " ")
        print(f"{i}. hybrid={hybrid_score:.3f} | dense={dense_score:.3f} | bm25={bm25_score:.3f} | source={source} | page={page}")
        print(f"   {preview}")

manual_query = "ZXVMAX-S上网日志有哪些功能？"
fixed_results = manual_hybrid_search(manual_query, dense_weight=0.6, bm25_weight=0.4)
print_manual_hybrid_results("教学版固定权重混合检索", fixed_results)


### 动态权重融合

动态权重是在固定权重的基础上，根据 query 特征自动调节 dense 与 BM25 的比例：

- 产品名、告警码、接口名、字段名等关键词明显的问题，提高 BM25 权重；
- 原因分析、优化建议、影响说明等语义问题，提高 dense 权重；
- 其他问题保持均衡。


In [ ]:
def choose_manual_hybrid_weights(question: str):
    keyword_patterns = [
        r"\d{4,}",
        r"ZXVMAX-S",
        r"接口|字段|告警|日志|NAT|IMSI|MSISDN|URL",
        r"有哪些|包括哪些|列出|表格",
    ]
    semantic_patterns = [
        r"为什么|原因|如何优化|怎么解决|影响|优缺点|解释",
    ]
    if any(re.search(pattern, question, re.IGNORECASE) for pattern in keyword_patterns):
        return 0.4, 0.6, "keyword_first"
    if any(re.search(pattern, question, re.IGNORECASE) for pattern in semantic_patterns):
        return 0.7, 0.3, "semantic_first"
    return 0.5, 0.5, "balanced"

for test_query in [
    "ZXVMAX-S上网日志有哪些功能？",
    "为什么需要做5GC业务质量定界？",
    "114020039告警如何处理？",
]:
    dense_weight, bm25_weight, query_type = choose_manual_hybrid_weights(test_query)
    results = manual_hybrid_search(
        test_query,
        dense_weight=dense_weight,
        bm25_weight=bm25_weight,
        final_k=3,
    )
    print(f"\nQuery: {test_query}")
    print(f"query_type={query_type}, dense_weight={dense_weight}, bm25_weight={bm25_weight}")
    print_manual_hybrid_results("教学版动态权重混合检索", results)


## Milvus 内置混合检索 + Rerank

上面的手写版本用于理解原理，但工程上不推荐把 BM25 索引拉到本地维护。Milvus 2.6 可以在服务端通过 `BM25BuiltInFunction` 生成 sparse 字段，并在搜索时直接做 weighted 或 RRF 融合。

下面开始使用上一章构建好的 `ZXVMAXS_HYBRID`，这才是推荐的实际调用方式。


## 加载混合检索向量库

这里加载上一章创建的 `ZXVMAXS_HYBRID` collection。它包含两个向量字段：

- `dense`：语义 embedding 字段；
- `sparse`：Milvus BM25 内置函数生成的关键词检索字段。


In [ ]:
from pymilvus import MilvusClient, connections
from langchain_milvus import Milvus, BM25BuiltInFunction
from langchain_ollama import OllamaEmbeddings

my_emb = OllamaEmbeddings(
    base_url="http://192.168.1.166:11434",
    model="qwen3-embedding:0.6b",
)

connection_args = {
    "uri": "http://8.159.144.108:19530",
}

_milvus_client = MilvusClient(**connection_args)
_milvus_alias = _milvus_client._using
if not connections.has_connection(_milvus_alias):
    connections.connect(alias=_milvus_alias, uri=connection_args["uri"])

vectordb_hybrid = Milvus(
    embedding_function=my_emb,
    builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
    collection_name="ZXVMAXS_HYBRID",
    connection_args=connection_args,
    vector_field=["dense", "sparse"],
    text_field="text",
    consistency_level="Bounded",
)

# 检索结果只返回文本和元数据字段，不返回 dense/sparse 向量原始值。
vectordb_hybrid.fields = [
    field for field in vectordb_hybrid.fields
    if field not in {"dense", "sparse", "vector"}
]

print("向量字段：", vectordb_hybrid.vector_fields)
print("返回字段：", vectordb_hybrid.fields)


## Weighted 混合检索

Weighted Ranker 会把 dense 与 sparse 两路分数按权重融合。下面设置 `weights=[0.6, 0.4]`：

- `0.6` 给 dense 语义检索；
- `0.4` 给 sparse BM25 关键词检索。

当问题更偏自然语言时，可以提高 dense 权重；当问题包含告警码、产品名、接口名等关键词时，可以提高 sparse 权重。


In [ ]:
def print_docs(title, docs, preview_chars=220):
    print(f"\n===== {title} =====")
    displayed = set()
    row = 1
    for doc in docs:
        source = str(doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源").strip()
        page = str(doc.metadata.get("page", "-")).strip()
        key = (source, page, doc.page_content[:80])
        if key in displayed:
            continue
        displayed.add(key)
        preview = doc.page_content[:preview_chars].replace("\n", " ")
        score = doc.metadata.get("rerank_score")
        score_text = f" | rerank_score={score:.2f}" if isinstance(score, (int, float)) else ""
        print(f"{row}. source={source} | page={page}{score_text}")
        print(f"   {preview}")
        row += 1

query = "ZXVMAX-S上网日志有哪些功能？"

weighted_docs = vectordb_hybrid.similarity_search(
    query,
    k=5,
    ranker_type="weighted",
    ranker_params={"weights": [0.6, 0.4]},
)

print_docs("Weighted 混合检索结果", weighted_docs)


## RRF 混合检索

RRF（Reciprocal Rank Fusion）不需要手动设置 dense/sparse 的权重，而是根据多个召回列表中的排序位置进行融合。它适合教学和初始实验：当你不确定权重如何设置时，可以先用 RRF 作为稳健基线。


In [ ]:
rrf_docs = vectordb_hybrid.similarity_search(
    query,
    k=5,
    ranker_type="rrf",
    ranker_params={"k": 100},
)

print_docs("RRF 混合检索结果", rrf_docs)


## 动态权重策略

实际业务中，不同问题适合不同权重：

- 包含“告警码、接口名、产品名、字段名、编号”等关键词的问题，适合提高 BM25 sparse 权重；
- 表达更自然、需要语义理解的问题，适合提高 dense 权重；
- 不确定时使用均衡权重，或者直接使用 RRF。

下面用一个简单规则演示动态权重。这个规则不是最终工程方案，但适合帮助学生理解权重调节的意义。


In [ ]:
import re

def choose_hybrid_weights(question: str):
    keyword_patterns = [
        r"\d{4,}",          # 长数字、告警码、编号
        r"ZXVMAX-S",          # 产品名
        r"接口|字段|告警|日志|NAT|IMSI|MSISDN|URL",
        r"有哪些|包括哪些|列出|表格",
    ]
    semantic_patterns = [
        r"为什么|原因|如何优化|怎么解决|影响|优缺点|解释",
    ]

    if any(re.search(pattern, question, re.IGNORECASE) for pattern in keyword_patterns):
        return [0.4, 0.6], "keyword_first"
    if any(re.search(pattern, question, re.IGNORECASE) for pattern in semantic_patterns):
        return [0.7, 0.3], "semantic_first"
    return [0.5, 0.5], "balanced"

for test_query in [
    "ZXVMAX-S上网日志有哪些功能？",
    "为什么需要做5GC业务质量定界？",
    "114020039告警如何处理？",
]:
    weights, query_type = choose_hybrid_weights(test_query)
    docs = vectordb_hybrid.similarity_search(
        test_query,
        k=3,
        ranker_type="weighted",
        ranker_params={"weights": weights},
    )
    print(f"\nQuery: {test_query}")
    print(f"query_type={query_type}, weights={weights}")
    print_docs("动态权重检索结果", docs, preview_chars=120)


## Rerank 精排

混合检索解决的是“尽量召回更多可能相关的候选文档”，但候选列表里仍可能有冗余或弱相关内容。Rerank 的作用是对 query 与候选文档逐对打分，再重新排序，通常用于：

1. 先用 hybrid search 召回较多候选，例如 top 10 或 top 20；
2. 再用 reranker 精排，保留 top 3 到 top 5；
3. 最后把精排后的高质量上下文交给 LLM。

本节提供三种 Rerank 方式：

- **Cohere Rerank**：云端 API，效果成熟，需要 `COHERE_API_KEY`；
- **Jina Reranker**：云端 HTTP API，支持多语言，需要 `JINA_API_KEY`；
- **本地 Qwen3 Reranker**：通过 Ollama 调用 `dengcao/Qwen3-Reranker-0.6B:Q8_0`，适合内网和教学环境。


### Cohere Rerank（仅示例）

如果环境中配置了 `COHERE_API_KEY`，可以使用 `langchain_cohere.CohereRerank`。没有 API Key 时跳过该单元即可。


In [ ]:
import os

if os.getenv("COHERE_API_KEY"):
    from langchain_cohere import CohereRerank

    cohere_compressor = CohereRerank(
        model="rerank-multilingual-v3.0",
        top_n=3,
    )
    cohere_reranked_docs = cohere_compressor.compress_documents(weighted_docs, query=query)
    print_docs("Cohere Rerank 结果", cohere_reranked_docs)
else:
    print("未配置 COHERE_API_KEY，跳过 Cohere Rerank 示例。")


### Jina Reranker（仅示例）

Jina Reranker 可以直接通过 HTTP API 调用。下面给出通用模板：如果环境中配置了 `JINA_API_KEY`，就会实际调用；否则只打印跳过信息。


In [ ]:
import os
import httpx
from langchain_core.documents import Document

def jina_rerank(query, documents, top_n=3, model="jina-reranker-v2-base-multilingual"):
    api_key = os.getenv("JINA_API_KEY")
    if not api_key:
        print("未配置 JINA_API_KEY，跳过 Jina Rerank 示例。")
        return []

    payload = {
        "model": model,
        "query": query,
        "top_n": top_n,
        "documents": [doc.page_content for doc in documents],
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    response = httpx.post(
        "https://api.jina.ai/v1/rerank",
        headers=headers,
        json=payload,
        timeout=60,
        trust_env=False,
    )
    response.raise_for_status()
    results = response.json().get("results", [])

    reranked = []
    for item in results:
        idx = item["index"]
        score = item.get("relevance_score", 0.0)
        doc = documents[idx]
        reranked.append(Document(
            page_content=doc.page_content,
            metadata={**doc.metadata, "rerank_score": float(score), "rerank_model": model},
        ))
    return reranked

jina_reranked_docs = jina_rerank(query, weighted_docs, top_n=3)
if jina_reranked_docs:
    print_docs("Jina Rerank 结果", jina_reranked_docs)


### 本地 Qwen3 Reranker（推荐教学环境）

本地方式不依赖外部 API Key，通过 Ollama 调用 `dengcao/Qwen3-Reranker-0.6B:Q8_0`。需要注意：这类 reranker 模型本质上更适合用 yes/no token 概率做相关性判断；在普通 ChatOllama 生成接口下，有时不会输出稳定的数字分数。因此下面的教学实现采用“模型优先、兜底可解释”的策略：

1. 优先解析模型输出中的数字、`yes/no` 或“相关/不相关”；
2. 如果输出无法解析，则使用 query 与文档的关键词/bigram 重叠分作为兜底，保证课堂演示稳定运行；
3. 生产环境建议使用支持 logits 或专用 rerank API 的服务封装。


In [ ]:
import re
from langchain_ollama import ChatOllama
from langchain_core.documents import Document

reranker_llm = ChatOllama(
    base_url="http://192.168.1.166:11434",
    model="dengcao/Qwen3-Reranker-0.6B:Q8_0",
    temperature=0,
    num_predict=8,
    sync_client_kwargs={"timeout": 20},
)

def _query_features(text: str):
    """提取用于兜底打分的简单特征：英文/数字词 + 中文 bigram。"""
    text = text.lower()
    tokens = set(re.findall(r"[a-z0-9][a-z0-9_-]+", text))
    chinese = "".join(re.findall(r"[\u4e00-\u9fff]+", text))
    tokens.update(chinese[i:i + 2] for i in range(max(len(chinese) - 1, 0)))
    return {token for token in tokens if len(token) >= 2}

def _fallback_overlap_score(query: str, document: str) -> float:
    """当 Ollama chat 接口无法返回可解析 rerank 分数时，使用可解释的关键词重叠分兜底。"""
    features = _query_features(query)
    if not features:
        return 0.0
    doc_text = document.lower()
    hit_count = sum(1 for feature in features if feature in doc_text)
    return round(100 * hit_count / len(features), 2)

def _parse_rerank_score(raw_output: str):
    text = (raw_output or "").strip().lower()
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    if match:
        return max(0.0, min(100.0, float(match.group()))), "model_numeric"
    if "yes" in text or "相关" in text:
        return 100.0, "model_yes_no"
    if "no" in text or "不相关" in text:
        return 0.0, "model_yes_no"
    return None, "fallback_overlap"

def local_qwen3_rerank(query, documents, top_n=3):
    scored_docs = []
    for doc in documents:
        prompt = f"""请判断文档与问题的相关性，只输出 0 到 100 的数字分数，不要输出解释。
问题：{query}
文档：{doc.page_content[:1200]}"""
        raw_output = ""
        try:
            response = reranker_llm.invoke(prompt)
            raw_output = getattr(response, "content", str(response))
            score, score_source = _parse_rerank_score(raw_output)
            if score is None:
                score = _fallback_overlap_score(query, doc.page_content)
        except Exception as exc:
            raw_output = f"ERROR: {exc}"
            score_source = "fallback_overlap_error"
            score = _fallback_overlap_score(query, doc.page_content)

        scored_docs.append(Document(
            page_content=doc.page_content,
            metadata={
                **doc.metadata,
                "rerank_score": score,
                "rerank_score_source": score_source,
                "rerank_raw_output": raw_output[:80],
                "rerank_model": "dengcao/Qwen3-Reranker-0.6B:Q8_0",
            },
        ))

    scored_docs.sort(key=lambda item: item.metadata.get("rerank_score", 0.0), reverse=True)
    return scored_docs[:top_n]


## 混合检索 + 本地 Rerank 实测

下面先用 weighted 混合检索召回 8 条候选，再用本地 Qwen3 Reranker 精排保留 3 条。教学时可以对比 rerank 前后的来源页和内容片段，观察精排是否把更贴近问题的上下文排到前面。


In [ ]:
rerank_query = "ZXVMAX-S上网日志有哪些功能？"

candidate_docs = vectordb_hybrid.similarity_search(
    rerank_query,
    k=3,
    ranker_type="weighted",
    ranker_params={"weights": [0.6, 0.4]},
)

print_docs("Rerank 前：混合检索候选", candidate_docs, preview_chars=140)

reranked_docs = local_qwen3_rerank(rerank_query, candidate_docs, top_n=3)
print_docs("Rerank 后：本地 Qwen3 精排结果", reranked_docs, preview_chars=180)


## 小结

本章的推荐实践路径是：

1. **基础版**：先用 dense 向量检索搭建可工作的 RAG；
2. **进阶版**：引入 Milvus BM25 sparse 字段，用 weighted 或 RRF 做混合检索；
3. **质量优化**：当召回结果多而杂时，在混合检索后加入 Rerank，把更相关的上下文排到前面；
4. **模型选择**：Cohere/Jina 适合云端 API 场景，本地 Qwen3 Reranker 适合内网、私有化和教学环境。

需要特别注意：Milvus 的 `sparse` 字段是检索字段，不是展示字段。无论是否加入 Rerank，最终返回给 LangChain 文档对象的都应该是 `text/source/page` 等普通字段。


## 混合检索问答链-完整版本（含记忆）

前面的混合检索和 Rerank 解决的是“召回和排序”的问题，但一个完整的 RAG 教学案例还需要把它们接成真正的问答链。

这一小节补上三件事：

1. **检索**：继续使用 `ZXVMAXS_HYBRID` 的 weighted 混合检索；
2. **生成**：用统一的回答模板组织上下文、问题和回答要求；
3. **记忆**：用 `RunnableWithMessageHistory` 保存多轮对话，让追问可以依赖前文。

### 这条链为什么更完整

和前面的单轮检索问答相比，这里多了一个“会话状态”：

- 第一轮问“ZXVMAX-S 上网日志有哪些功能？”，系统先检索，再生成答案；
- 第二轮问“那它支持保存多久？”，系统会把前一轮对话带入当前回答，避免把“它”理解错；
- 如果课堂上要展示内部过程，可以直接打印历史和改写后的检索问题。

这类写法更接近真实业务里的连续问答助手。

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# 复用前面章节里已经创建好的混合库；如果当前会话里没有，再按统一配置重建一次。
qa_hybrid_db = globals().get("vectordb_hybrid")
if qa_hybrid_db is None:
    from pymilvus import MilvusClient, connections
    from langchain_milvus import Milvus, BM25BuiltInFunction
    from langchain_ollama import OllamaEmbeddings

    qa_connection_args = {
        "uri": "http://8.159.144.108:19530",
    }
    qa_emb = OllamaEmbeddings(
        base_url="http://192.168.1.166:11434",
        model="qwen3-embedding:0.6b",
    )
    _qa_milvus_client = MilvusClient(**qa_connection_args)
    _qa_alias = _qa_milvus_client._using
    if not connections.has_connection(_qa_alias):
        connections.connect(alias=_qa_alias, uri=qa_connection_args["uri"])

    qa_hybrid_db = Milvus(
        embedding_function=qa_emb,
        builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
        collection_name="ZXVMAXS_HYBRID",
        connection_args=qa_connection_args,
        vector_field=["dense", "sparse"],
        text_field="text",
        consistency_level="Bounded",
    )

# 复用已经建好的混合检索库；重新过滤一次字段，避免 sparse/dense 被当作原始数据返回。
qa_hybrid_db.fields = [
    field for field in qa_hybrid_db.fields
    if field not in {"dense", "sparse", "vector"}
]

# 复用前面章节里已经创建好的 LLM；如果当前会话里没有，再按统一模型配置重建一次。
qa_llm = globals().get("my_llm")
if qa_llm is None:
    import os
    import httpx
    from dotenv import load_dotenv
    from langchain.chat_models import init_chat_model

    load_dotenv("/Users/will/workspace/ai-showcase/.env")
    api_key = os.getenv("SILICONFLOW_API_KEY")
    if not api_key:
        raise RuntimeError("未配置 SILICONFLOW_API_KEY，无法运行检索问答链示例。")

    qa_llm = init_chat_model(
        model="Qwen/Qwen3-8B",
        model_provider="openai",
        base_url="https://api.siliconflow.cn/v1",
        api_key=api_key,
        http_client=httpx.Client(trust_env=False),
        temperature=0.7,
        max_tokens=10000,
    )

rewrite_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "你是一个检索问题改写器。请结合对话历史，把当前问题改写成一个独立、完整、适合检索的问题。只输出改写后的问题，不要解释。",
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "你是 VMAX 运维助手。只基于给定上下文回答，不要编造。"
        "如果上下文没有提供答案，明确回答“我不知道”。"
        "回答末尾补充引用来源和页码。",
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    (
        "human",
        "原始问题：{question}\n\n"
        "改写后的检索问题：{standalone_question}\n\n"
        "相关上下文：\n{context}\n\n"
        "请给出简洁、准确的中文回答。",
    ),
])

question_rewriter = rewrite_prompt | qa_llm | StrOutputParser()


def _format_retrieval_docs(docs, preview_chars=260):
    lines = []
    for i, doc in enumerate(docs, 1):
        source = str(doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源")
        page = str(doc.metadata.get("page", "-"))
        preview = doc.page_content[:preview_chars].replace("\n", " ")
        lines.append(f"[{i}] source={source} | page={page}\n{preview}")
    return "\n\n".join(lines)


def build_hybrid_qa_inputs(inputs: dict):
    question = inputs["question"]
    chat_history = inputs.get("chat_history", [])

    standalone_question = question
    if chat_history:
        standalone_question = question_rewriter.invoke({
            "question": question,
            "chat_history": chat_history,
        }).strip()

    docs = qa_hybrid_db.similarity_search(
        standalone_question,
        k=6,
        ranker_type="weighted",
        ranker_params={"weights": [0.6, 0.4]},
    )

    if "local_qwen3_rerank" in globals():
        docs = local_qwen3_rerank(standalone_question, docs, top_n=3)

    return {
        "question": question,
        "standalone_question": standalone_question,
        "chat_history": chat_history,
        "context": _format_retrieval_docs(docs),
    }

hybrid_qa_chain = (
    RunnableLambda(build_hybrid_qa_inputs)
    | answer_prompt
    | qa_llm
    | StrOutputParser()
)

conversation_store = {}


def get_hybrid_qa_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in conversation_store:
        conversation_store[session_id] = ChatMessageHistory()
    return conversation_store[session_id]

hybrid_qa_with_memory = RunnableWithMessageHistory(
    hybrid_qa_chain,
    get_hybrid_qa_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

print("混合检索问答链已准备完成。")
print("- 支持 weighted 混合检索")
print("- 支持可选本地 Rerank")
print("- 支持会话记忆")


### 先看单轮效果

先跑一轮常见的功能类问题，确认完整问答链可以直接工作。

In [ ]:
qa_session_id = "hybrid_qa_demo"
first_question = "ZXVMAX-S上网日志有哪些功能？"
first_answer = hybrid_qa_with_memory.invoke(
    {"question": first_question},
    config={"configurable": {"session_id": qa_session_id}},
)

print("用户问题：", first_question)
print("\n模型回答：\n")
print(first_answer)

### 再看带记忆的追问

第二轮问题故意不把主语说完整，观察系统是否会利用前一轮对话把问题补全后再检索。

In [ ]:
follow_up_question = "那它支持保存多久？"

# 先把当前会话里的历史取出来，顺便展示改写后的检索问题。
current_history = get_hybrid_qa_history(qa_session_id).messages
preview_inputs = {
    "question": follow_up_question,
    "chat_history": current_history,
}
preview_result = build_hybrid_qa_inputs(preview_inputs)

print("用户追问：", follow_up_question)
print("改写后的检索问题：", preview_result["standalone_question"])
print("\n检索上下文预览：\n")
print(preview_result["context"][:1400])

second_answer = hybrid_qa_with_memory.invoke(
    {"question": follow_up_question},
    config={"configurable": {"session_id": qa_session_id}},
)

print("\n模型回答：\n")
print(second_answer)

### 这一版链路的教学价值

这一节把“混合检索”补成了真正可对话的 RAG 链：

- 检索层负责召回和排序；
- 改写层负责把追问变成可检索的问题；
- 记忆层负责保留上下文；
- 生成层负责把检索内容组织成稳定答案。

到这里，混合检索章节就不是只有检索对比，而是完整覆盖了“召回 - 精排 - 生成 - 记忆”的闭环。

# RAG进阶-Agentic RAG

前面的基础版 RAG 和混合检索 RAG 都是“开发者固定流程”：用户问题一定会进入某一个检索链，然后由 LLM 基于检索结果回答。

Agentic RAG 的思路是把检索能力包装成工具，让 Agent 根据问题自己判断：

```text
用户问题 -> Agent 判断任务类型 -> 选择工具 -> 观察工具结果 -> 生成最终答案
```

这和 `03_LangChain制作智能体.ipynb` 中的 Agent 思路一致，只是这里把普通工具升级成 RAG 工具：

- **简单检索工具**：基于 `ZXVMAXS`，只做 dense 语义检索，适合语义型问题；
- **混合检索工具**：基于 `ZXVMAXS_HYBRID`，融合 dense + BM25，适合产品名、模块名、告警码、专有名词较多的问题；
- **计算工具**：用于演示 Agent 不一定只会查知识库，也可以选择其他工具完成任务。

教学时可以把它理解为：RAG 是一种能力，Agent 是调度这些能力的决策层。

## Agentic RAG 的工具设计

工具设计的关键不是“把所有逻辑都塞给 Agent”，而是把每个工具的边界写清楚：

| 工具 | 适合问题 | 底层能力 |
|---|---|---|
| `simple_rag_search` | 语义描述类问题、概念解释类问题 | Milvus dense 向量检索 |
| `hybrid_rag_search` | 产品名、模块名、告警码、接口名、字段名等关键词明显的问题 | Milvus dense + BM25 混合检索，可接 Rerank |
| `calculate` | 数字计算、简单表达式计算 | Python AST 白名单计算 |

工具的 docstring 会进入模型上下文，直接影响 Agent 是否会选对工具。因此工具说明要具体、克制、可区分。

## 准备 Agentic RAG 工具

下面的代码会重新连接两个已经构建好的 Milvus 集合，保证这一节可以单独运行：

- `ZXVMAXS`：简单 dense 检索集合；
- `ZXVMAXS_HYBRID`：混合检索集合，包含 `dense` 和 `sparse` 两个向量字段。

注意：Milvus 的向量字段只用于检索，不应该作为原始字段返回，所以这里仍然会过滤 `dense/sparse/vector` 字段。

In [ ]:
from pymilvus import MilvusClient, connections
from langchain_milvus import Milvus, BM25BuiltInFunction
from langchain_ollama import OllamaEmbeddings
from langchain_core.tools import tool

# 这一节独立重建连接，方便从 Agentic RAG 章节开始单独运行。
agent_connection_args = {
    "uri": "http://localhost:19530",
}

agent_emb = OllamaEmbeddings(
    base_url="http://192.168.1.166:11434",
    model="qwen3-embedding:0.6b",
)

_agent_milvus_client = MilvusClient(**agent_connection_args)
_agent_milvus_alias = _agent_milvus_client._using
if not connections.has_connection(_agent_milvus_alias):
    connections.connect(alias=_agent_milvus_alias, uri=agent_connection_args["uri"])

agent_simple_db = Milvus(
    embedding_function=agent_emb,
    collection_name="ZXVMAXS",
    connection_args=agent_connection_args,
    vector_field="dense",
    text_field="text",
    consistency_level="Bounded",
)
agent_simple_db.fields = [
    field for field in agent_simple_db.fields
    if field not in {"dense", "sparse", "vector"}
]

agent_hybrid_db = Milvus(
    embedding_function=agent_emb,
    builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
    collection_name="ZXVMAXS_HYBRID",
    connection_args=agent_connection_args,
    vector_field=["dense", "sparse"],
    text_field="text",
    consistency_level="Bounded",
)
agent_hybrid_db.fields = [
    field for field in agent_hybrid_db.fields
    if field not in {"dense", "sparse", "vector"}
]

def _format_tool_docs(docs, max_chars=320):
    """把检索结果压缩成适合工具返回给 Agent 的文本。"""
    lines = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源"
        page = doc.metadata.get("page", "-")
        content = doc.page_content[:max_chars].replace("\n", " ")
        lines.append(f"[{i}] source={source} | page={page}\n{content}")
    return "\n\n".join(lines)

@tool
def simple_rag_search(question: str) -> str:
    """使用简单 dense 向量检索查询 ZXVMAX-S 知识库。适合语义型、描述型、概念解释类问题。"""
    docs = agent_simple_db.similarity_search(question, k=4)
    if not docs:
        return "简单检索没有找到相关资料。"
    return _format_tool_docs(docs)

@tool
def hybrid_rag_search(question: str) -> str:
    """使用 Milvus 混合检索查询 ZXVMAX-S 知识库。适合包含产品名、模块名、告警码、接口名、字段名等关键词的问题。"""
    docs = agent_hybrid_db.similarity_search(
        question,
        k=4,
        ranker_type="weighted",
        ranker_params={"weights": [0.5, 0.5]},
    )
    if "local_qwen3_rerank" in globals():
        docs = local_qwen3_rerank(question, docs, top_n=4)
    if not docs:
        return "混合检索没有找到相关资料。"
    return _format_tool_docs(docs)

@tool
def calculate(expression: str) -> str:
    """安全计算数学表达式。适合四则运算、乘方和括号表达式。不要用于知识库问答。"""
    import ast
    import operator

    operators = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Pow: operator.pow,
        ast.USub: operator.neg,
    }

    def safe_eval(node):
        if isinstance(node, ast.Expression):
            return safe_eval(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in operators:
            return operators[type(node.op)](safe_eval(node.left), safe_eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in operators:
            return operators[type(node.op)](safe_eval(node.operand))
        raise ValueError(f"不支持的表达式：{ast.dump(node)}")

    try:
        tree = ast.parse(expression, mode="eval")
        return f"计算结果：{expression} = {safe_eval(tree)}"
    except Exception as exc:
        return f"计算错误：{exc}"

agentic_rag_tools = [simple_rag_search, hybrid_rag_search, calculate]

print("Agentic RAG 工具准备完成：")
for item in agentic_rag_tools:
    print("-", item.name, ":", item.description)

## 先直接测试工具

在正式交给 Agent 调度之前，先直接调用工具本身。这样做有两个好处：

1. 确认工具和外部系统连接正常；
2. 让学生看到“Agent 调用工具”之前，工具返回的原始内容是什么。

如果工具本身不可用，Agent 再聪明也无法得到正确结果。

In [ ]:
tool_test_questions = [
    (simple_rag_search, {"question": "为什么需要做5GC业务质量定界？"}),
    (hybrid_rag_search, {"question": "ZXVMAX-S上网日志有哪些功能？"}),
    (calculate, {"expression": "(15 * 37 + 228) / 3"}),
]

for tool_item, args in tool_test_questions:
    print("=" * 80)
    print("工具：", tool_item.name)
    print("输入：", args)
    print(tool_item.invoke(args)[:1200])

## 构建工具调用 Agent

当前环境使用的是 LangChain 1.x，新版推荐用 `create_agent()` 创建 Agent 图；旧教程中常见的 `AgentExecutor`、`create_tool_calling_agent` 在部分新版本里已经不再从 `langchain.agents` 直接导出。

它和普通 RAG 链的最大区别是：

- 普通 RAG 链：开发者提前决定一定走哪个 retriever；
- Agentic RAG：模型先理解问题，再决定是否调用工具、调用哪个工具、如何利用工具结果回答。

默认关闭 `debug`，输出更适合教材阅读。如果课堂上想观察模型消息、工具调用和工具返回结果，可以把代码中的 `DEBUG_AGENTIC_RAG` 改为 `True`。

In [ ]:
import os
import httpx
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

load_dotenv("/Users/will/workspace/ai-showcase/.env")
agent_api_key = os.getenv("SILICONFLOW_API_KEY")

agentic_system_prompt = (
    "你是一个企业知识库问答助手。"
    "你可以使用工具查询 ZXVMAX-S 知识库或完成计算。"
    "当问题包含产品名、模块名、告警码、接口名、字段名等明确关键词时，优先使用 hybrid_rag_search。"
    "当问题更偏语义描述、原因解释、概念说明时，可以使用 simple_rag_search。"
    "当问题是数学计算时，使用 calculate。"
    "回答必须基于工具返回内容；如果工具没有提供依据，要明确说明知识库中没有找到充分依据。"
    "知识库问答的答案最后要列出引用来源和页码。"
)

if not agent_api_key:
    print("未配置 SILICONFLOW_API_KEY，跳过 Agent 模型调用；上面的工具直测仍可用于理解 Agentic RAG。")
else:
    agentic_llm = init_chat_model(
        model="Qwen/Qwen3-8B",
        model_provider="openai",
        base_url="https://api.siliconflow.cn/v1",
        api_key=agent_api_key,
        http_client=httpx.Client(trust_env=False),
        temperature=0,
        max_tokens=4000,
    )

    DEBUG_AGENTIC_RAG = False

    agentic_agent = create_agent(
        model=agentic_llm,
        tools=agentic_rag_tools,
        system_prompt=agentic_system_prompt,
        debug=DEBUG_AGENTIC_RAG,
    )

    print("Agentic RAG Agent 已创建。")

## Agentic RAG 效果测试

这里准备三类问题，观察 Agent 是否能选对工具：

1. **关键词明显的问题**：应优先走混合检索；
2. **语义解释问题**：可以走简单检索；
3. **计算问题**：应走计算工具，而不是知识库检索。

真实项目中，Agent 的工具选择不应只靠一次演示判断，需要结合 LangSmith、日志或评测集持续观察。

In [ ]:
agentic_examples = [
    "ZXVMAX-S上网日志有哪些功能？",
    "为什么需要做5GC业务质量定界？",
    "帮我计算 (15 * 37 + 228) / 3。",
]

def print_agent_answer(result):
    """兼容 LangChain 1.x create_agent 返回的 messages 结构。"""
    messages = result.get("messages", []) if isinstance(result, dict) else []
    if not messages:
        print(result)
        return
    final_message = messages[-1]
    content = getattr(final_message, "content", final_message)
    print(content)

if "agentic_agent" not in globals():
    print("Agentic Agent 尚未创建，通常是因为没有配置 SILICONFLOW_API_KEY。")
else:
    for question in agentic_examples:
        print("=" * 80)
        print("用户问题：", question)
        response = agentic_agent.invoke({
            "messages": [{"role": "user", "content": question}],
        })
        print("最终回答：")
        print_agent_answer(response)

## Agentic RAG 小结

Agentic RAG 不是替代普通 RAG，而是在普通 RAG 已经稳定后增加一层“工具调度能力”。建议按下面顺序教学和落地：

1. **先做普通 RAG**：保证检索、提示词和回答质量可控；
2. **再做混合检索 + Rerank**：提升召回与排序质量；
3. **最后引入 Agent**：把不同检索链、计算工具、业务 API 封装成工具，让模型按任务选择；
4. **持续评估工具选择**：Agent 的核心风险不是代码能不能跑，而是它是否在正确场景选择正确工具。

一句话总结：普通 RAG 解决“如何基于知识库回答”，Agentic RAG 解决“什么时候查、查哪里、是否还要调用其他工具”。

# RAG 效果评估

RAG 链路搭完之后，最后一步是评估它是否真的比“裸大模型”更可靠。

本章使用 **LLM-as-Judge** 的思路，做一个轻量级对比实验：

1. 准备 3 个 VMAX 领域的测试问题，并给出每个问题的参考答案要点；
2. 分别用 **带 RAG 的混合检索问答链** 和 **直接调用大模型（无知识库）** 生成答案；
3. 让另一个 LLM 从 **忠实度、相关性、完整性** 三个维度打分（1-5 分，5 分最好）；
4. 用表格对比两种方案的分数，观察 RAG 对幻觉抑制和知识准确性的作用。

> 说明：这里不引入额外评估库（如 Ragas），只使用已经配置好的大模型做裁判，便于在 `env_rag` 环境中直接运行。分数仅供参考，真实业务中建议结合人工标注。

In [ ]:
# 评估环境准备：复用统一模型配置，连接混合检索向量库
import os
import httpx
import re
import pandas as pd
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_ollama import OllamaEmbeddings
from langchain_milvus import Milvus, BM25BuiltInFunction
from pymilvus import MilvusClient, connections
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

load_dotenv("/Users/will/workspace/ai-showcase/.env")
api_key = os.getenv("SILICONFLOW_API_KEY")
if not api_key:
    raise ValueError("请先配置 SILICONFLOW_API_KEY")

# 裁判/生成模型：温度设为 0，评分更稳定
eval_llm = init_chat_model(
    model="Qwen/Qwen3-8B",
    model_provider="openai",
    base_url="https://api.siliconflow.cn/v1",
    api_key=api_key,
    http_client=httpx.Client(trust_env=False),
    temperature=0,
    max_tokens=2000,
)

eval_emb = OllamaEmbeddings(
    base_url="http://192.168.1.166:11434",
    model="qwen3-embedding:0.6b",
)

connection_args = {"uri": "http://8.159.144.108:19530"}
_eval_milvus_client = MilvusClient(**connection_args)
_eval_alias = _eval_milvus_client._using
if not connections.has_connection(_eval_alias):
    connections.connect(alias=_eval_alias, uri=connection_args["uri"])

eval_hybrid_db = Milvus(
    embedding_function=eval_emb,
    builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
    collection_name="ZXVMAXS_HYBRID",
    connection_args=connection_args,
    vector_field=["dense", "sparse"],
    text_field="text",
    consistency_level="Bounded",
)
# 检索时不返回向量原始字段
eval_hybrid_db.fields = [
    field for field in eval_hybrid_db.fields
    if field not in {"dense", "sparse", "vector"}
]

def _format_eval_context(docs, preview_chars=300):
    lines = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源"
        page = doc.metadata.get("page", "-")
        content = doc.page_content[:preview_chars].replace("\n", " ")
        lines.append(f"[{i}] source={source} | page={page}\n{content}")
    return "\n\n".join(lines)

def retrieve_for_eval(question: str, k: int = 5):
    docs = eval_hybrid_db.similarity_search(
        question,
        k=k,
        ranker_type="weighted",
        ranker_params={"weights": [0.6, 0.4]},
    )
    return _format_eval_context(docs)

print("评估环境准备完成。")


In [ ]:
# 构造 RAG 问答链与基线问答链
rag_prompt = PromptTemplate.from_template("""
你是 VMAX 运维助手，请严格根据以下上下文回答问题。

上下文：
{context}

问题：{question}

回答要求：
1. 只基于上下文回答，不要编造上下文之外的信息；
2. 如果上下文没有提供答案，请回答“我不知道”；
3. 回答简洁、准确。

回答：""")

rag_answer_chain = (
    {"context": lambda q: retrieve_for_eval(q), "question": RunnablePassthrough()}
    | rag_prompt
    | eval_llm
    | StrOutputParser()
)

baseline_answer_chain = eval_llm | StrOutputParser()

# 评估数据集：问题 + 参考答案要点
eval_set = [
    {
        "question": "ZXVMAX-S上网日志有哪些功能？",
        "reference": "上网日志保存、上网日志查询、批量导入查询、日志管理、账号/角色管理、资源监控、告警管理、省级网关对接、拨测结果自动比对、NAT日志入库、北向接口、上网日志历史查询、云化上网日志XDR查询",
    },
    {
        "question": "什么是ZXVMAX-S的5GC业务？",
        "reference": "面向5G核心网的多维价值分析系统；包含5G精准开通、5G业务运营、一键投诉、区域感知、用户感知、实时跟踪/会话跟踪等功能",
    },
    {
        "question": "114020039告警如何处理？",
        "reference": "NodeManager进程本地空间使用率告警；可能原因为Yarn空间使用率超过阈值；处理步骤包括修改告警阈值、磁盘扩容、收集日志联系技术支持",
    },
]

print(f"准备评估 {len(eval_set)} 个问题")


In [ ]:
# 生成答案（这一步会调用 LLM 和 Milvus，可能需要几十秒）
for item in eval_set:
    item["rag_answer"] = rag_answer_chain.invoke(item["question"])
    item["baseline_answer"] = baseline_answer_chain.invoke(item["question"])

print("RAG 与基线答案已生成完成。")


In [ ]:
# LLM-as-Judge：对答案进行三维度评分
judge_prompt = PromptTemplate.from_template("""
你是一名公正的评估专家。请根据下面的上下文、问题和参考答案要点，对助手的回答进行评分。

【上下文】
{context}

【问题】
{question}

【参考答案要点】
{reference}

【助手回答】
{answer}

请从以下三个维度打分（1-5 分，5 分最好），并给出简短理由：
1. 忠实度：回答内容是否完全能从上下文中找到依据。
2. 相关性：回答是否直接针对问题，没有答非所问。
3. 完整性：回答是否覆盖了参考答案要点。

请严格按以下格式输出：
忠实度: <分数>
相关性: <分数>
完整性: <分数>
理由: <一句话理由>
""")

judge_chain = judge_prompt | eval_llm | StrOutputParser()

def parse_judge_result(text: str):
    result = {"faithfulness": 0, "relevance": 0, "completeness": 0, "reason": ""}
    faith = re.search(r"忠实度[:：]\s*(\d+)", text)
    rel = re.search(r"相关性[:：]\s*(\d+)", text)
    comp = re.search(r"完整性[:：]\s*(\d+)", text)
    reason = re.search(r"理由[:：]\s*(.+)", text, re.DOTALL)
    if faith:
        result["faithfulness"] = int(faith.group(1))
    if rel:
        result["relevance"] = int(rel.group(1))
    if comp:
        result["completeness"] = int(comp.group(1))
    if reason:
        result["reason"] = reason.group(1).strip().replace("\n", " ")
    return result

for item in eval_set:
    context = retrieve_for_eval(item["question"])
    for key in ["rag_answer", "baseline_answer"]:
        judge_text = judge_chain.invoke({
            "context": context,
            "question": item["question"],
            "reference": item["reference"],
            "answer": item[key],
        })
        item[f"{key}_score"] = parse_judge_result(judge_text)

print("评分完成。")


In [ ]:
# 汇总对比结果
rows = []
for item in eval_set:
    rows.append({
        "问题": item["question"],
        "方案": "RAG",
        "忠实度": item["rag_answer_score"]["faithfulness"],
        "相关性": item["rag_answer_score"]["relevance"],
        "完整性": item["rag_answer_score"]["completeness"],
        "平均": round(sum([item["rag_answer_score"][k] for k in ["faithfulness", "relevance", "completeness"]]) / 3, 2),
    })
    rows.append({
        "问题": item["question"],
        "方案": "基线（无RAG）",
        "忠实度": item["baseline_answer_score"]["faithfulness"],
        "相关性": item["baseline_answer_score"]["relevance"],
        "完整性": item["baseline_answer_score"]["completeness"],
        "平均": round(sum([item["baseline_answer_score"][k] for k in ["faithfulness", "relevance", "completeness"]]) / 3, 2),
    })

df = pd.DataFrame(rows)
display(df)

print("\n===== 平均分对比 =====")
print(df.groupby("方案")[["忠实度", "相关性", "完整性", "平均"]].mean().round(2))

print("\n===== 详细理由示例（第一个问题） =====")
print("问题：", eval_set[0]["question"])
print("\n[RAG 回答]")
print(eval_set[0]["rag_answer"][:300])
print("理由：", eval_set[0]["rag_answer_score"]["reason"])
print("\n[基线回答]")
print(eval_set[0]["baseline_answer"][:300])
print("理由：", eval_set[0]["baseline_answer_score"]["reason"])


## 评估结果解读

通过这个简单的 LLM-as-Judge 实验，通常可以观察到：

- **RAG 方案**：忠实度和完整性通常明显高于基线，因为答案来自本地知识库；
- **基线方案**：容易出现“看似专业、实则编造”的回答（比如把 ZXVMAX-S 说成华为产品），幻觉风险更高；
- **相关性**：如果问题表述清晰，两者通常都不错；但如果问题需要文档中的具体关键词，RAG 更稳。

## 局限性与后续优化

1. **LLM 裁判也有偏差**：评分模型本身可能偏好“更长”或“更完整”的答案，结果仅供参考；
2. **样本量小**：这里只用了 3 个问题，真实业务建议准备 50+ 条评测集；
3. **没有评估检索质量**：可以进一步加入 `Recall@K`、`MRR` 等检索指标；
4. **可以引入 Ragas**：如果希望更系统，可以安装 `ragas`，用内置的 `faithfulness`、`answer_relevancy`、`context_recall` 等指标。

这个章节把 RAG 从“能跑通”推进到了“可评估”，让整个 Notebook 形成了一个更完整的闭环。

## 延伸阅读：Ragas 评估框架

**Ragas**（Retrieval-Augmented Generation Assessment）是专为 RAG 系统设计的一个开源评估框架，用于在不需要人工标注参考答案的情况下，对检索和生成两个环节进行量化评估。

### 核心指标

- **Faithfulness（忠实度）**：衡量生成答案是否在提供的上下文中能找到依据，检测模型是否"编造"了上下文中不存在的信息。

- **Answer Relevancy（答案相关性）**：衡量生成答案与问题的相关程度，惩罚答非所问或包含无关冗余内容的回答。

- **Context Relevancy（上下文相关性）**：衡量检索到的文档片段与问题的相关程度，去除无关内容的比例。

- **Context Recall（上下文召回率）**：衡量检索到的上下文是否包含了回答问题的全部必要信息。

### 为什么使用 Ragas

1. **自动化**：无需人工准备标准答案，适合快速迭代 RAG 链路；
2. **端到端**：同时覆盖检索和生成，便于定位是"召回不够"还是"生成不准"；
3. **与 LangChain 集成友好**：可以直接传入 LangChain 的 `retriever` 和 `chain` 结果进行打分。

### 使用思路

在真实业务场景中，建议先用本节 LLM-as-Judge 的方法跑通评估流程，再逐步迁移到 Ragas。典型步骤是：准备 50 条以上的评测问题 → 用 LangChain 分别得到问题和答案 → 调用 Ragas 内置指标计算平均分 → 通过拆解"检索指标"和"生成指标"定位优化方向。

通过引入 Ragas，RAG 系统不再只是"能回答"，而是可以持续、可量化地"回答得更好"。